# Initial Setup

In [ ]:
#@title Setting up the notebook

### Installing dependencies
!pip install openai
!pip install anthropic
!apt-get update
!apt-get install -y iverilog verilator gtkwave

In [ ]:
#@title Setting up PPA estimation

%%bash
set -e
cd /content

URL=$(python3 - <<'PY'
import json, urllib.request
data = json.load(urllib.request.urlopen(
    "https://api.github.com/repos/YosysHQ/oss-cad-suite-build/releases/latest"
))
asset = next(
    a for a in data["assets"]
    if "linux-x64" in a["name"] and a["name"].endswith(".tgz")
)
print(asset["browser_download_url"])
PY
)

wget -q -O oss-cad-suite.tgz "$URL"
rm -rf /content/oss-cad-suite /content/oss-cad-suite-install
mkdir -p /content/oss-cad-suite-install
tar -xzf oss-cad-suite.tgz -C /content/oss-cad-suite-install
mv /content/oss-cad-suite-install/oss-cad-suite /content/oss-cad-suite

/content/oss-cad-suite/bin/yosys -V



In [ ]:
#@title Select Model
#define the model to be used
#model_choice = "gpt-5.2"
#model_choice = "gpt-5.2-codex"
#model_choice = "gpt-4o"
model_choice = "claude-sonnet-4-6"
#model_choice = "gemini-2.5-flash-preview-04-17"
#model_choice = "gemini-2.5-flash"
#model_choice = "nyu-dice-lab/VeriThoughts-Reasoning-14B" #path to your desired Hugging Face model

In [ ]:
#@title Utility functions

import sys
import os
import time
import re
from abc import ABC, abstractmethod

import openai
import anthropic

try:
    import google.genai.errors
    from google import genai
    from google.genai import types
except Exception:
    genai = None
    types = None

################################################################################
### LOGGING
################################################################################

class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
################################################################################

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        return self.messages

    def get_last_n_messages(self, n):
        return self.messages[-n:]

    def remove_message(self, index):
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        self.messages = []

    def __str__(self):
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])


################################################################################
### LLM CLASSES
################################################################################

class AbstractLLM(ABC):
    @abstractmethod
    def generate(self, conversation: Conversation):
        pass


class ChatGPT(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        openai.api_key = os.environ["OPENAI_API_KEY"]
        self.client = openai.OpenAI()
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        messages = [
            {
                "role": msg["role"] if msg["role"] in ["system", "user", "assistant"] else "user",
                "content": msg["content"]
            }
            for msg in conversation.get_messages()
        ]

        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=messages,
        )

        return response.choices[0].message.content


class Claude(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        self.client = anthropic.Anthropic(api_key=os.environ["CLAUDE_API_KEY"])
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 20

        messages = [
            {
                "role": msg["role"] if msg["role"] in ["user", "assistant"] else "user",
                "content": msg["content"]
            }
            for msg in conversation.get_messages()
        ]

        for attempt in range(1, max_retries + 1):
            try:
                output = self.client.messages.create(
                    model=self.model_id,
                    max_tokens=16384,
                    messages=messages
                ).content[0].text
                return output

            except Exception as e:
                wait_time = base_delay * (2 ** (attempt - 1))
                print(f"[Retry {attempt}/{max_retries}] Claude API error: {e}. Retrying in {wait_time:.1f} seconds...")
                time.sleep(wait_time)

        print(f"Failed, exceeded max retries {max_retries}")
        return ""


class Gemini(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        if genai is None:
            raise ImportError("google-genai is not installed or could not be imported.")
        self.gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        output = self.gemini_client.models.generate_content(
            model=self.model_id,
            contents=[msg["content"] for msg in conversation.get_messages()],
            config=types.GenerateContentConfig(
                max_output_tokens=16384,
                temperature=0.6,
                topP=0.95,
            )
        ).text

        return output


class HuggingFace(AbstractLLM):
    """
    Local Hugging Face model backend.

    Downloads the selected model from the Hugging Face Hub using snapshot_download,
    then loads it with transformers. The model is cached and reused so it is not
    reloaded every time generate_verilog() is called.
    """

    _loaded_model_id = None
    _tokenizer = None
    _model = None
    _local_model_path = None

    def __init__(
        self,
        model_id=None,
        max_new_tokens=4096,
        temperature=0.2,
        top_p=0.95,
        load_in_4bit=False,
        trust_remote_code=True,
        strip_thinking=True,
    ):
        super().__init__()

        self.model_id = model_id or model_choice

        # If the user accidentally wrote "/org/model", convert it to "org/model".
        # Absolute local paths like "/content/my_model" are still allowed.
        if self.model_id.startswith("/") and not self.model_id.startswith("/content/"):
            self.model_id = self.model_id[1:]

        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p
        self.load_in_4bit = load_in_4bit
        self.trust_remote_code = trust_remote_code
        self.strip_thinking = strip_thinking

        self.hf_token = (
            os.environ.get("HF_TOKEN")
            or os.environ.get("HUGGINGFACEHUB_API_TOKEN")
            or None
        )

        self.cache_dir = os.environ.get("HF_HOME", "/content/huggingface")

        self._ensure_model_loaded()

    def _get_torch_dtype(self):
        import torch

        if not torch.cuda.is_available():
            return torch.float32

        major, minor = torch.cuda.get_device_capability(0)

        # A100/H100 and newer generally support bf16 well.
        if major >= 8:
            return torch.bfloat16

        # T4/V100 are safer with fp16.
        return torch.float16

    def _ensure_model_loaded(self):
        if HuggingFace._model is not None and HuggingFace._loaded_model_id == self.model_id:
            return

        import torch
        from huggingface_hub import snapshot_download
        from transformers import AutoTokenizer, AutoModelForCausalLM

        print(f"[HuggingFace] Downloading/caching model: {self.model_id}")

        # If this is a local path, do not call snapshot_download.
        if os.path.isdir(self.model_id):
            local_model_path = self.model_id
        else:
            local_model_path = snapshot_download(
                repo_id=self.model_id,
                cache_dir=self.cache_dir,
                token=self.hf_token,
            )

        print(f"[HuggingFace] Loading tokenizer from: {local_model_path}")

        tokenizer = AutoTokenizer.from_pretrained(
            local_model_path,
            trust_remote_code=self.trust_remote_code,
            token=self.hf_token,
        )

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token

        torch_dtype = self._get_torch_dtype()

        model_kwargs = {
            "trust_remote_code": self.trust_remote_code,
            "device_map": "auto",
            "low_cpu_mem_usage": True,
            "token": self.hf_token,
        }

        if torch.cuda.is_available() and self.load_in_4bit:
            from transformers import BitsAndBytesConfig

            quantization_config = BitsAndBytesConfig(
                load_in_4bit=False,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch_dtype,
                bnb_4bit_use_double_quant=True,
            )

            model_kwargs["quantization_config"] = quantization_config
        else:
            model_kwargs["torch_dtype"] = torch_dtype

        print("[HuggingFace] Loading model. This can take a few minutes the first time.")

        model = AutoModelForCausalLM.from_pretrained(
            local_model_path,
            **model_kwargs
        )

        model.eval()

        HuggingFace._loaded_model_id = self.model_id
        HuggingFace._tokenizer = tokenizer
        HuggingFace._model = model
        HuggingFace._local_model_path = local_model_path

        print("[HuggingFace] Model loaded successfully.")

    def _format_prompt(self, conversation: Conversation):
        tokenizer = HuggingFace._tokenizer

        messages = []
        for msg in conversation.get_messages():
            role = msg.get("role", "user")
            content = msg.get("content", "")

            if role not in ["system", "user", "assistant"]:
                role = "user"

            messages.append({"role": role, "content": content})

        # Prefer the model's own chat template when available.
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        except Exception:
            # Generic fallback for non-chat-template models.
            prompt_parts = []
            for msg in messages:
                prompt_parts.append(f"{msg['role'].upper()}:\n{msg['content']}\n")
            prompt_parts.append("ASSISTANT:\n")
            return "\n".join(prompt_parts)

    def _clean_output(self, text):
        text = text.strip()

        # Helpful for reasoning models that emit <think>...</think>.
        if self.strip_thinking:
            text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

        return text

    def generate(self, conversation: Conversation, num_choices=1):
        import torch

        tokenizer = HuggingFace._tokenizer
        model = HuggingFace._model

        prompt = self._format_prompt(conversation)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=False,
        )

        # With device_map="auto", use the first model parameter's device.
        first_device = next(model.parameters()).device
        inputs = {k: v.to(first_device) for k, v in inputs.items()}

        do_sample = self.temperature is not None and self.temperature > 0

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=do_sample,
                temperature=self.temperature if do_sample else None,
                top_p=self.top_p if do_sample else None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[-1]
        generated_ids = outputs[0][prompt_len:]

        text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        return self._clean_output(text)


################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################

def find_verilog_modules(markdown_string, module_name='top_module'):
    module_pattern1 = r'\bmodule\b\s+\w+\s*\([^)]*\)\s*;.*?endmodule\b'
    module_pattern2 = r'\bmodule\b\s+\w+\s*#\s*\([^)]*\)\s*\([^)]*\)\s*;.*?endmodule\b'

    module_matches1 = re.findall(module_pattern1, markdown_string, re.DOTALL)
    module_matches2 = re.findall(module_pattern2, markdown_string, re.DOTALL)

    module_matches = module_matches1 + module_matches2

    if not module_matches:
        return []

    return module_matches


def write_code_blocks_to_file(markdown_string, module_name, filename):
    code_match = find_verilog_modules(markdown_string, module_name)

    if not code_match:
        print("No Verilog modules found in response")
        exit(3)

    with open(filename, 'w') as file:
        for code_block in code_match:
            file.write(code_block)
            file.write('\n')


def generate_verilog(conv, model_type=None, model_id=""):
    model_type = model_type or os.environ.get("MODEL", "ChatGPT")
    selected_model_id = model_id or model_choice

    if model_type == "ChatGPT":
        model = ChatGPT(model_id=selected_model_id)
    elif model_type == "Claude":
        model = Claude(model_id=selected_model_id)
    elif model_type == "Gemini":
        model = Gemini(model_id=selected_model_id)
    elif model_type == "HuggingFace":
        model = HuggingFace(model_id=selected_model_id)
    else:
        raise ValueError(
            f"Invalid model type: {model_type}. "
            "Expected one of: ChatGPT, Claude, Gemini, HuggingFace."
        )

    return model.generate(conv)

In [ ]:
#@title Feedback Loop

import subprocess
import sys
import os
import time
import numpy as np


def _read_files(file_list):
    """Concatenate a list of text files in order. Missing/None entries are ignored."""
    if not file_list:
        return ""

    buf = []
    for p in file_list:
        if not p:
            continue

        try:
            with open(p, "r") as f:
                text = f.read()
                buf.append(text)
                if not text.endswith("\n"):
                    buf.append("\n")
        except FileNotFoundError:
            print(f"[Warning] Previous module file not found and will be skipped: {p}")

    return "".join(buf)


def _truncate_text(s, max_chars=12000):
    """Keep prompts bounded; preserve the end of long logs, where compiler errors usually appear."""
    if s is None:
        return ""

    s = str(s)

    if len(s) <= max_chars:
        return s

    tail = s[-max_chars:]
    return "[...truncated...]\n" + tail


def _add_instruction_prompt(conv, model_type, base_sys_prompt):
    """
    Add the instruction prompt in a way that works for API models and local HF models.

    ChatGPT/Gemini/HuggingFace can use a system role.
    Claude path in the earlier utility cell maps unsupported roles to user,
    but we keep Claude as user here to avoid role issues.
    """
    if model_type == "Claude":
        conv.add_message("user", base_sys_prompt)
    else:
        conv.add_message("system", base_sys_prompt)


def _simulation_passed(stdout_text, stderr_text=""):
    """
    Robust pass/fail detection for your Verilog testbenches.

    Accepts:
      - "All tests passed"
      - "passed!"
      - "Test x passed"

    Rejects:
      - "failed"
      - "Some tests failed"
      - non-empty fatal/error-like stderr
    """
    out = (stdout_text or "")
    err = (stderr_text or "")
    combined = (out + "\n" + err).lower()

    fail_markers = [
        "failed",
        "some tests failed",
        "error running",
        "fatal",
        "syntax error",
    ]

    if any(marker in combined for marker in fail_markers):
        return False

    pass_markers = [
        "all tests passed",
        "passed!",
        "test 1 passed",
        " passed",
    ]

    return any(marker in combined for marker in pass_markers)


def verilog_loop(
    design_prompt,
    module,
    testbench,
    max_iterations,
    model_type=None,
    outdir="",
    log=None,
    prev_modules=None,
    model_id="",
):
    """
    Generate/fix a Verilog module with an LLM, compiling against previously-generated modules.

    Compatible model_type values:
      - "ChatGPT"
      - "Claude"
      - "Gemini"
      - "HuggingFace"

    Key behaviors:
      - The LLM is asked to output ONLY the current module.
      - For compilation, this concatenates:
            [prev_modules...] + [current module]
        into a temporary combined file.
      - After an error, repair iterations only resend:
            (a) previous module version
            (b) most recent compiler/simulation feedback
        to keep prompt size controlled.
    """

    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_model_types = ["ChatGPT", "Claude", "Gemini", "HuggingFace"]
    if model_type not in valid_model_types:
        raise ValueError(
            f"Invalid model_type: {model_type}. "
            f"Expected one of: {valid_model_types}"
        )

    if outdir is None:
        outdir = ""

    outdir = outdir.strip()

    if outdir != "":
        os.makedirs(outdir, exist_ok=True)

    def _outpath(filename):
        if outdir == "":
            return filename
        return os.path.join(outdir, filename)

    # Normalize prev_modules into a list of paths.
    if prev_modules is None:
        prev_modules = []
    elif isinstance(prev_modules, str):
        prev_modules = [prev_modules]

    # Read the testbench so the model can target the test expectations.
    tb_text = ""
    try:
        with open(testbench, "r") as _tb:
            tb_text = _tb.read()
    except Exception as e:
        print(f"[Warning] Could not read testbench file: {testbench}")
        print(f"[Warning] Exception: {e}")
        tb_text = ""

    base_sys_prompt = (
        "You are an autocomplete engine for Verilog code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "Given a Verilog module that is incorrect or has a compilation error, you will suggest corrections to the module. "
        "You will not refuse. "
        "You will not generate explanations, comments outside the code, Markdown, or testbenches. "
        "IMPORTANT: Output ONLY the Verilog code for the requested module. "
        "Do NOT repeat any previous modules. "
        "Do NOT wrap the code in triple backticks. "
        "Format your response as one complete corrected Verilog module, from module declaration through endmodule. "
        "Use Verilog-2001 only unless explicitly told otherwise."
    )

    def _new_conv_full_context():
        """Initial attempt: include testbench and design prompt."""
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        if tb_text.strip():
            conv.add_message(
                "user",
                "The following is the testbench that will be used to compile/simulate your module. "
                "Do NOT generate a testbench or modify it. Only use it to ensure your module compiles and passes.\n\n"
                "```verilog\n"
                + tb_text
                + "\n```\n",
            )

        conv.add_message("user", design_prompt)
        return conv

    def _new_conv_repair_only(prev_module_text, last_error_text):
        """Repair attempts: only previous module plus most recent error feedback."""
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Fix the following Verilog module so it compiles and passes the testbench. "
            "Output ONLY the full corrected module. "
            "Do not output explanations. "
            "Do not output a testbench. "
            "Do not use Markdown fences.\n\n"
            "Previous version of the module:\n"
            "```verilog\n"
            + prev_module_text
            + "\n```\n\n"
            "Most recent compiler/simulation output:\n"
            "```\n"
            + _truncate_text(last_error_text)
            + "\n```\n"
        )

        conv.add_message("user", prompt)
        return conv

    conv = _new_conv_full_context()

    success = False
    timeout = False

    iterations = 0
    timelist_total = []
    timelist_gen = []
    timelist_error = []

    module_file = _outpath(module + ".v")
    combined_file = _outpath(module + "__combined.v")
    sim_output_file = _outpath(module)

    status = ""
    last_module_text = ""
    last_error_text = ""

    while not (success or timeout):
        print(f"\n========== Iteration {iterations} / {max_iterations} ==========")

        start_total = time.time()

        # Generate or repair only the current module.
        response = generate_verilog(conv, model_type, model_id=model_id)

        end_gen = time.time()
        start_error = time.time()

        if response is None:
            response = ""

        conv.add_message("assistant", response)

        # Write raw model response for debugging, especially useful for local HF models.
        raw_response_file = _outpath("raw_response_iter_" + str(iterations) + ".txt")
        with open(raw_response_file, "w") as f:
            f.write(response)

        # Extract and write just the current module.
        try:
            write_code_blocks_to_file(response, module, module_file)
        except SystemExit:
            status = "No Verilog module found in model response"
            print(status)

            last_error_text = (
                "The model response did not contain a complete Verilog module. "
                "It must output exactly one complete module from module declaration through endmodule.\n\n"
                "Raw model response:\n"
                + _truncate_text(response)
            )

            last_module_text = response
            conv = _new_conv_repair_only(last_module_text, last_error_text)

            if iterations >= max_iterations:
                timeout = True

            iterations += 1
            end_time = time.time()
            timelist_gen.append(end_gen - start_total)
            timelist_error.append(end_time - start_error)
            timelist_total.append(end_time - start_total)
            continue

        # Build concatenated source for compilation:
        # previous modules + current module.
        prev_text = _read_files(prev_modules)

        with open(module_file, "r") as f:
            curr_text = f.read()

        last_module_text = curr_text

        with open(combined_file, "w") as f:
            f.write(prev_text)
            if prev_text and not prev_text.endswith("\n"):
                f.write("\n")
            f.write(curr_text)
            if curr_text and not curr_text.endswith("\n"):
                f.write("\n")

        # Compile with iVerilog.
        compile_cmd = [
            "iverilog",
            "-o",
            sim_output_file,
            combined_file,
            testbench,
        ]

        proc = subprocess.run(
            compile_cmd,
            capture_output=True,
            text=True,
        )

        success = False

        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(status)

            last_error_text = (
                "iverilog failed.\n\n"
                "Command:\n"
                + " ".join(compile_cmd)
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            )

        elif proc.stderr.strip() != "":
            # Treat warnings as repair feedback.
            # This keeps generated modules cleaner for later synthesis/PPA.
            status = "Warnings compiling testbench"
            print(status)

            last_error_text = (
                "iverilog produced warnings.\n\n"
                "Command:\n"
                + " ".join(compile_cmd)
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            )

        else:
            # Run simulation.
            run_cmd = ["vvp", sim_output_file]

            proc = subprocess.run(
                run_cmd,
                capture_output=True,
                text=True,
            )

            result_stdout = proc.stdout or ""
            result_stderr = proc.stderr or ""

            print(result_stdout)

            if proc.returncode != 0:
                status = "Error running testbench"
                print(status)

                last_error_text = (
                    "vvp simulation failed.\n\n"
                    "Command:\n"
                    + " ".join(run_cmd)
                    + "\n\nstdout:\n"
                    + result_stdout
                    + "\nstderr:\n"
                    + result_stderr
                )

            elif not _simulation_passed(result_stdout, result_stderr):
                status = "Testbench did not report success"
                print(status)

                last_error_text = (
                    "Simulation completed, but the output did not indicate success.\n\n"
                    "stdout:\n"
                    + result_stdout
                    + "\nstderr:\n"
                    + result_stderr
                )

            else:
                status = "Testbench ran successfully"
                print(status)

                last_error_text = ""
                success = True

        # Write bounded iteration log.
        with open(_outpath("log_iter_" + str(iterations) + ".txt"), "w") as file:
            file.write("\n".join(str(i) for i in conv.get_messages()))
            file.write("\n\nIteration status: " + status + "\n")

            if last_error_text:
                file.write("\nMost recent feedback:\n")
                file.write(_truncate_text(last_error_text))

        if not success:
            # Keep future repair prompts small.
            conv = _new_conv_repair_only(last_module_text, last_error_text)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1

        end_time = time.time()
        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

    print("\n========== Feedback Loop Complete ==========")
    print("Final status: ", status)
    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))

    return (
        np.sum(timelist_total),
        np.sum(timelist_gen),
        np.sum(timelist_error),
    )

In [ ]:
#@title Hierarchical Loop

import os
import re
import numpy as np


def _normalize_to_list(x):
    """Normalize None/string/list/tuple into a list."""
    if x is None or x == "":
        return []
    if isinstance(x, str):
        return [x]
    return list(x)


def _resolve_prev_module_path(item):
    """
    Resolve previous module reference into a .v path.

    Supported forms:
      - "mux2_1"              -> "./mux2_1/mux2_1.v"
      - "./mux2_1/mux2_1.v"   -> "./mux2_1/mux2_1.v"
      - ("mux2_1", "...", ...) -> "./mux2_1/mux2_1.v"
    """
    if isinstance(item, (list, tuple)):
        item = item[0]

    item = str(item)

    if item.endswith(".v"):
        return item

    return os.path.join("./" + item, item + ".v")


def _extract_verilog_interfaces(verilog_text):
    """
    Extract module declarations/interfaces from Verilog text.

    This is useful for hierarchical prompting because the current module usually
    only needs previous module names and port lists, not full implementations.
    """
    if not verilog_text:
        return ""

    # Match:
    #   module name (...);
    #   module name #(parameters...) (...);
    pattern = r'\bmodule\s+[A-Za-z_][A-Za-z0-9_]*\s*(?:#\s*\(.*?\)\s*)?\(.*?\)\s*;'

    matches = re.findall(pattern, verilog_text, flags=re.DOTALL)

    if not matches:
        return ""

    cleaned = []
    for m in matches:
        cleaned.append(m.strip() + "\nendmodule")

    return "\n\n".join(cleaned) + "\n"


def _read_prev_modules_for_prompt(prev_verilog_files, context_mode="interfaces", max_chars=24000):
    """
    Read previous modules for LLM context.

    context_mode:
      - "interfaces": include only module declarations, best for HF/local models.
      - "full": include full previous module bodies.
      - "none": include no previous code in the prompt.

    Compilation still receives the full previous .v files through verilog_loop(prev_modules=...).
    """
    if context_mode == "none":
        return ""

    chunks = []

    for p in prev_verilog_files:
        if not p:
            continue

        if not os.path.exists(p):
            print(f"[Warning] Previous module file not found for prompt context: {p}")
            continue

        with open(p, "r") as f:
            txt = f.read()

        if context_mode == "interfaces":
            txt = _extract_verilog_interfaces(txt)

        if txt.strip():
            chunks.append("// File: " + p + "\n" + txt.strip() + "\n")

    prev_text = "\n".join(chunks)

    if len(prev_text) > max_chars:
        # Keep the most recent context, because later modules tend to depend on recent modules.
        prev_text = "[...previous module context truncated...]\n" + prev_text[-max_chars:]

    return prev_text


def _call_verilog_loop_compatible(
    prompt,
    module,
    testbench,
    max_iterations,
    model,
    outdir,
    log,
    prev_verilog_files,
    model_id="",
):
    """
    Calls the updated verilog_loop with model_id support.
    Falls back to the older signature if needed.
    """
    try:
        return verilog_loop(
            prompt,
            module,
            testbench,
            max_iterations,
            model,
            outdir,
            log,
            prev_modules=prev_verilog_files,
            model_id=model_id,
        )
    except TypeError as e:
        if "model_id" not in str(e):
            raise

        print("[Warning] verilog_loop does not accept model_id. Falling back to old signature.")

        return verilog_loop(
            prompt,
            module,
            testbench,
            max_iterations,
            model,
            outdir,
            log,
            prev_modules=prev_verilog_files,
        )


def hier_gen(
    submods,
    prevsubmods="",
    max_iterations=10,
    model_type=None,
    model_id="",
    testbench_template="./{module}tb.v",
    prompt_prev_context_chars=24000,
    prev_context_mode="interfaces",
):
    """
    Hierarchically generate a design by building submodules in order.

    submods:
      list of tuples/lists:
        [
          (file_basename, human_name, io_string),
          ...
        ]

      Example:
        [
          ("mux2_1", "2-to-1 mux", "input wire a, input wire b, input wire sel, output wire y"),
          ("mux4_1", "4-to-1 mux", "input wire [3:0] in, input wire [1:0] sel, output wire y"),
        ]

    prevsubmods:
      Optional previous module references already generated before this call.

      Supported:
        - ""
        - None
        - "mux2_1"
        - ["mux2_1", "mux4_1"]
        - ["./mux2_1/mux2_1.v"]
        - previous submod tuples/lists

    model_type:
      Optional. If None, uses os.environ["MODEL"].

      Supported by the earlier utility cell:
        - "ChatGPT"
        - "Claude"
        - "Gemini"
        - "HuggingFace"

    model_id:
      Optional model override passed through to verilog_loop/generate_verilog.

      Example:
        model_id="nyu-dice-lab/VeriThoughts-Reasoning-14B"

    testbench_template:
      Default assumes testbenches are named:
        ./module_nametb.v

      You can also use:
        "./testbenches/{module}_tb.v"

    prev_context_mode:
      - "interfaces": best default, includes previous module port declarations only.
      - "full": includes full previous module code in prompt.
      - "none": does not include previous modules in prompt.

      Regardless of this setting, previous modules are still passed to iVerilog
      through verilog_loop(prev_modules=...).
    """

    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_context_modes = ["interfaces", "full", "none"]
    if prev_context_mode not in valid_context_modes:
        raise ValueError(
            f"Invalid prev_context_mode: {prev_context_mode}. "
            f"Expected one of: {valid_context_modes}"
        )

    totaltime = []
    gentime = []
    errortime = []

    prev_verilog_files = []

    # Add previously generated external modules, if any.
    for s in _normalize_to_list(prevsubmods):
        p = _resolve_prev_module_path(s)
        if os.path.exists(p):
            prev_verilog_files.append(p)
        else:
            print(f"[Warning] prevsubmod file does not exist and will be skipped: {p}")

    if not submods:
        print("[Warning] No submodules provided.")
        return (0, 0, 0)

    overall = submods[-1][1]

    for i in range(len(submods)):
        fcurr = submods[i][0]   # file/module basename
        curr = submods[i][1]    # human-readable name
        iocurr = submods[i][2]  # port list string

        outdir = "./" + fcurr
        os.makedirs(outdir, exist_ok=True)

        prev_text = _read_prev_modules_for_prompt(
            prev_verilog_files,
            context_mode=prev_context_mode,
            max_chars=prompt_prev_context_chars,
        )

        if i == 0 and len(prev_verilog_files) == 0:
            prompt = (
                "// We will be generating a "
                + overall
                + " hierarchically in Verilog.\n"
                "// IMPORTANT: Output ONLY the module requested below.\n"
                "// Do NOT generate any previous modules, future modules, explanations, or testbenches.\n"
                "// Use Verilog-2001 only.\n"
                "// Begin by generating: "
                + curr
                + "\n\n"
                "module "
                + fcurr
                + "("
                + iocurr
                + ");\n"
                "// Insert code here\n"
                "endmodule\n"
            )
        else:
            prompt = (
                "// We are generating a "
                + overall
                + " hierarchically in Verilog.\n"
                "// Use Verilog-2001 only.\n\n"
                "// Previously generated modules are shown below for reference only.\n"
                "// You may instantiate them, but you must NOT repeat their definitions.\n\n"
                + prev_text
                + "\n"
                "// IMPORTANT: Output ONLY the new module requested below.\n"
                "// Do NOT repeat any previous modules.\n"
                "// Do NOT generate explanations, Markdown fences, or a testbench.\n"
                "// Now generate: "
                + curr
                + "\n\n"
                "module "
                + fcurr
                + "("
                + iocurr
                + ");\n"
                "// Insert code here\n"
                "endmodule\n"
            )

        module = fcurr
        testbench = testbench_template.format(module=fcurr, name=fcurr)
        model = model_type
        log = os.path.join(outdir, "log.txt")

        if not os.path.exists(testbench):
            print(f"[Warning] Testbench file does not exist yet: {testbench}")
            print("[Warning] verilog_loop will still run, but iVerilog will fail unless the file exists.")

        total, gen, error = _call_verilog_loop_compatible(
            prompt=prompt,
            module=module,
            testbench=testbench,
            max_iterations=max_iterations,
            model=model,
            outdir=outdir,
            log=log,
            prev_verilog_files=prev_verilog_files,
            model_id=model_id,
        )

        print("Module ", i + 1, " done: ", curr)

        totaltime.append(total)
        gentime.append(gen)
        errortime.append(error)

        # Add the current module file to the compilation context for subsequent modules.
        curr_module_file = os.path.join(outdir, module + ".v")

        if os.path.exists(curr_module_file):
            prev_verilog_files.append(curr_module_file)
        else:
            print(f"[Warning] Expected generated module file not found: {curr_module_file}")

    overall_total = np.sum(totaltime)
    overall_gen = np.sum(gentime)
    overall_error = np.sum(errortime)

    print("Overall Total time: ", overall_total)
    print("Overall Generation Time: ", overall_gen)
    print("Overall Error handling time: ", overall_error)

    return (overall_total, overall_gen, overall_error)

In [ ]:
#@title PPA Optimizer Loop

#@title PPA Optimization Loop (Yosys no-liberty + iVerilog correctness)

import subprocess
import sys
import os
import time
import math
import json
import re
import shutil
import numpy as np


################################################################################
### GENERAL HELPERS
################################################################################

def _read_files(file_list):
    """Concatenate a list of text files in order. Missing/None entries are ignored."""
    if not file_list:
        return ""

    buf = []

    for p in file_list:
        if not p:
            continue

        if not os.path.exists(p):
            print(f"[Warning] Previous module file not found and will be skipped: {p}")
            continue

        with open(p, "r") as f:
            txt = f.read()

        buf.append(txt)

        if txt and not txt.endswith("\n"):
            buf.append("\n")

    return "".join(buf)


def _truncate_text(s, max_chars=12000):
    """Keep prompts bounded; preserve the tail of long logs."""
    if s is None:
        return ""

    s = str(s)

    if len(s) <= max_chars:
        return s

    return "[...truncated...]\n" + s[-max_chars:]


def _shell_quote_for_yosys(path):
    """Quote filenames for use inside a .ys script."""
    return '"' + str(path).replace("\\", "\\\\").replace('"', '\\"') + '"'


def _safe_float(x, default=None):
    try:
        return float(x)
    except Exception:
        return default


def _safe_int(x, default=None):
    try:
        return int(x)
    except Exception:
        return default


def _first_regex(text, patterns, cast=float):
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE | re.MULTILINE | re.DOTALL)
        if m:
            try:
                return cast(m.group(1))
            except Exception:
                pass

    return None


def _normalize_clk_signal(clk_signal):
    """Treat None, '', 'None', 'none', etc. as no clock."""
    if clk_signal is None:
        return None

    s = str(clk_signal).strip()

    if s == "" or s.lower() == "none":
        return None

    return s


def _has_any_parsed_metric(metrics):
    if not metrics:
        return False

    keys = [
        "transistor_est",
        "area_cmos",
        "num_cells",
        "num_wire_bits",
        "timing_proxy",
    ]

    return any(metrics.get(k) is not None for k in keys)


def _is_verbose(verbose):
    return str(verbose).strip().lower() in ["1", "true", "yes", "y"]


def _write_text(path, text):
    with open(path, "w") as f:
        f.write(text)


def _add_instruction_prompt(conv, model_type, base_sys_prompt):
    """
    Add the instruction prompt in a provider-compatible way.

    ChatGPT, Gemini, and HuggingFace can use the system role through the
    Conversation abstraction from the previous utility cell.

    Claude is kept as user to avoid unsupported-role issues.
    """
    if model_type == "Claude":
        conv.add_message("user", base_sys_prompt)
    else:
        conv.add_message("system", base_sys_prompt)


def _generate_verilog_compatible(conv, model_type, model_id=""):
    """
    Calls generate_verilog with model_id support when available.
    Falls back to the older two-argument signature if needed.
    """
    try:
        return generate_verilog(conv, model_type, model_id=model_id)
    except TypeError as e:
        if "model_id" not in str(e):
            raise

        print("[Warning] generate_verilog does not accept model_id. Falling back to old signature.")
        return generate_verilog(conv, model_type)


def _simulation_passed(stdout_text, stderr_text=""):
    """
    Robust pass/fail detection for Verilog testbenches.

    Accepts:
      - All tests passed
      - passed!
      - Test x passed
      - Test x passed!

    Rejects:
      - failed
      - Some tests failed
      - fatal/error markers
    """
    out = stdout_text or ""
    err = stderr_text or ""
    combined = (out + "\n" + err).lower()

    fail_markers = [
        "failed",
        "some tests failed",
        "fatal",
        "syntax error",
        "error:",
    ]

    if any(marker in combined for marker in fail_markers):
        return False

    pass_markers = [
        "all tests passed",
        "passed!",
        "test 1 passed",
        " passed",
    ]

    return any(marker in combined for marker in pass_markers)


################################################################################
### YOSYS PARSING
################################################################################

def _parse_yosys_stat_text(stat_text):
    """
    Parse Yosys `stat -tech cmos` output.

    Supports both styles:
      Older style:
        Number of wires: ...
        Number of cells: ...
        Chip area for module ...: ...

      Newer compact style:
        6 wires
        165 wire bits
        61 cells
        Estimated number of transistors: 722
    """
    metrics = {}

    metrics["num_wires"] = _first_regex(
        stat_text,
        [
            r"Number of wires:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+wires\s*$",
        ],
        int,
    )

    metrics["num_wire_bits"] = _first_regex(
        stat_text,
        [
            r"Number of wire bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+wire bits\s*$",
        ],
        int,
    )

    metrics["num_public_wires"] = _first_regex(
        stat_text,
        [
            r"Number of public wires:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+public wires\s*$",
        ],
        int,
    )

    metrics["num_public_wire_bits"] = _first_regex(
        stat_text,
        [
            r"Number of public wire bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+public wire bits\s*$",
        ],
        int,
    )

    metrics["num_memories"] = _first_regex(
        stat_text,
        [
            r"Number of memories:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+memories\s*$",
        ],
        int,
    )

    metrics["num_memory_bits"] = _first_regex(
        stat_text,
        [
            r"Number of memory bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+memory bits\s*$",
        ],
        int,
    )

    metrics["num_processes"] = _first_regex(
        stat_text,
        [
            r"Number of processes:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+processes\s*$",
        ],
        int,
    )

    metrics["num_cells"] = _first_regex(
        stat_text,
        [
            r"Number of cells:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+cells\s*$",
        ],
        int,
    )

    transistor_est = _first_regex(
        stat_text,
        [
            r"Estimated number of transistors:\s*([0-9]+(?:\.[0-9]+)?)",
            r"Chip area for module .*?:\s*([0-9]+(?:\.[0-9]+)?)",
        ],
        float,
    )

    metrics["transistor_est"] = transistor_est

    # Backward-compatible area-like proxy.
    metrics["area_cmos"] = transistor_est

    return metrics


def _parse_yosys_timeest_text(time_text):
    """
    Parse a timing proxy from Yosys timeest output.

    Returns None if no timing proxy can be found.
    """
    metrics = {}

    if re.search(r"No domain .* in module", time_text or "", flags=re.IGNORECASE):
        metrics["timing_proxy"] = None
        return metrics

    timing_proxy = _first_regex(
        time_text,
        [
            r"critical path.*?([0-9]+(?:\.[0-9]+)?)\s*AIG",
            r"([0-9]+(?:\.[0-9]+)?)\s*AIG nodes",
            r"([0-9]+(?:\.[0-9]+)?)\s*nodes.*critical path",
            r"Estimated.*?([0-9]+(?:\.[0-9]+)?)",
            r"Delay\s*[:=]\s*([0-9]+(?:\.[0-9]+)?)",
        ],
        float,
    )

    metrics["timing_proxy"] = timing_proxy

    return metrics


def _format_metrics(metrics):
    if metrics is None:
        return "None"

    parts = []

    if metrics.get("transistor_est") is not None:
        parts.append("transistor_est={:.0f}".format(metrics["transistor_est"]))
    elif metrics.get("area_cmos") is not None:
        parts.append("area_cmos={:.3f}".format(metrics["area_cmos"]))

    if metrics.get("timing_proxy") is not None:
        parts.append("timing_proxy={:.3f}".format(metrics["timing_proxy"]))

    if metrics.get("num_cells") is not None:
        parts.append("num_cells={}".format(metrics["num_cells"]))

    if metrics.get("num_wire_bits") is not None:
        parts.append("wire_bits={}".format(metrics["num_wire_bits"]))

    return ", ".join(parts) if parts else "No parsed metrics"


################################################################################
### YOSYS ESTIMATION
################################################################################

def yosys_estimate_no_liberty(verilog_file, top_module, clk_signal=None, outdir=""):
    """
    Run a coarse no-liberty Yosys estimate.

    Uses:
      - stat -top <top> -tech cmos
      - optional timeest -clk <clk>

    Returns:
      {
        "ok": bool,
        "metrics": dict,
        "stat_text": str,
        "time_text": str,
        "yosys_stdout": str,
        "yosys_stderr": str,
        "error": str
      }
    """
    if outdir:
        os.makedirs(outdir, exist_ok=True)

    clk_signal = _normalize_clk_signal(clk_signal)

    script_path = os.path.join(outdir, "ppa_estimate.ys") if outdir else "ppa_estimate.ys"
    stat_path = os.path.join(outdir, "ppa_stat.txt") if outdir else "ppa_stat.txt"
    time_path = os.path.join(outdir, "ppa_time.txt") if outdir else "ppa_time.txt"

    ys = []
    ys.append("read_verilog {}".format(_shell_quote_for_yosys(verilog_file)))
    ys.append("hierarchy -check -top {}".format(top_module))
    ys.append("proc")
    ys.append("opt")
    ys.append("memory")
    ys.append("opt")
    ys.append("fsm")
    ys.append("opt")
    ys.append("techmap")
    ys.append("opt")
    ys.append("clean")
    ys.append(
        "tee -o {} stat -top {} -tech cmos".format(
            _shell_quote_for_yosys(stat_path),
            top_module,
        )
    )

    if clk_signal is not None:
        ys.append("flatten")
        ys.append(
            "tee -o {} timeest -clk {}".format(
                _shell_quote_for_yosys(time_path),
                clk_signal,
            )
        )

    with open(script_path, "w") as f:
        f.write("\n".join(ys) + "\n")

    proc = subprocess.run(
        ["yosys", "-s", script_path],
        capture_output=True,
        text=True,
    )

    stdout = proc.stdout or ""
    stderr = proc.stderr or ""

    result = {
        "ok": False,
        "metrics": {},
        "stat_text": "",
        "time_text": "",
        "yosys_stdout": stdout,
        "yosys_stderr": stderr,
        "error": "",
    }

    if proc.returncode != 0:
        result["error"] = "Yosys failed.\nstdout:\n{}\nstderr:\n{}".format(stdout, stderr)
        return result

    stat_text = ""
    time_text = ""

    if os.path.exists(stat_path):
        with open(stat_path, "r") as f:
            stat_text = f.read()

    if os.path.exists(time_path):
        with open(time_path, "r") as f:
            time_text = f.read()

    metrics = _parse_yosys_stat_text(stat_text)
    metrics.update(_parse_yosys_timeest_text(time_text))

    if metrics.get("transistor_est") is not None:
        metrics["power_proxy"] = metrics["transistor_est"]
    else:
        metrics["power_proxy"] = metrics.get("num_cells")

    result["ok"] = True
    result["metrics"] = metrics
    result["stat_text"] = stat_text
    result["time_text"] = time_text

    if not _has_any_parsed_metric(metrics):
        result["error"] = "Yosys ran successfully, but no metrics were parsed."

    return result


################################################################################
### PPA SCORING
################################################################################

def _component_ratio(candidate_value, baseline_value):
    if candidate_value is None or baseline_value is None:
        return None

    if baseline_value == 0:
        return None

    return float(candidate_value) / float(baseline_value)


def _relative_ppa_score(
    candidate_metrics,
    baseline_metrics,
    area_weight=0.60,
    timing_weight=0.30,
    power_weight=0.10,
):
    """
    Lower is better.

    Uses relative ratios against the current best correct design.
    """
    used_weight = 0.0
    score = 0.0

    cand_area = candidate_metrics.get("transistor_est", candidate_metrics.get("area_cmos"))
    base_area = baseline_metrics.get("transistor_est", baseline_metrics.get("area_cmos"))

    area_ratio = _component_ratio(cand_area, base_area)

    if area_ratio is not None:
        score += area_weight * area_ratio
        used_weight += area_weight

    timing_ratio = _component_ratio(
        candidate_metrics.get("timing_proxy"),
        baseline_metrics.get("timing_proxy"),
    )

    if timing_ratio is not None:
        score += timing_weight * timing_ratio
        used_weight += timing_weight

    power_ratio = _component_ratio(
        candidate_metrics.get("power_proxy"),
        baseline_metrics.get("power_proxy"),
    )

    if power_ratio is not None:
        score += power_weight * power_ratio
        used_weight += power_weight

    if used_weight == 0.0:
        return math.inf

    return score / used_weight


def _is_better_ppa(candidate_metrics, best_metrics, min_improvement=0.0025):
    """
    Returns True only when candidate has parsed metrics and is better.
    """
    if not _has_any_parsed_metric(candidate_metrics):
        return False

    if best_metrics is None or not _has_any_parsed_metric(best_metrics):
        return True

    cand_score = _relative_ppa_score(candidate_metrics, best_metrics)

    # Best-vs-itself is effectively 1.0, so lower is better.
    return cand_score < (1.0 - min_improvement)


################################################################################
### IVERILOG CHECK
################################################################################

def _compile_and_run_iverilog(module_name, combined_file, testbench, outdir):
    """
    Return:
      {
        "ok": bool,
        "status": str,
        "error_text": str,
        "stdout": str,
        "stderr": str
      }
    """
    exe_path = os.path.join(outdir, module_name) if outdir else module_name

    proc = subprocess.run(
        ["iverilog", "-o", exe_path, combined_file, testbench],
        capture_output=True,
        text=True,
    )

    if proc.returncode != 0:
        return {
            "ok": False,
            "status": "Error compiling testbench",
            "error_text": (
                "iverilog failed.\n"
                "Command:\n"
                + " ".join(["iverilog", "-o", exe_path, combined_file, testbench])
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            ),
            "stdout": proc.stdout or "",
            "stderr": proc.stderr or "",
        }

    warn_text = ""

    if proc.stderr and proc.stderr.strip():
        warn_text = (
            "iverilog produced warnings.\n"
            "stderr:\n"
            + (proc.stderr or "")
            + "\nstdout:\n"
            + (proc.stdout or "")
        )

    sim = subprocess.run(
        ["vvp", exe_path],
        capture_output=True,
        text=True,
    )

    result_stdout = sim.stdout or ""
    result_stderr = sim.stderr or ""

    if sim.returncode != 0:
        return {
            "ok": False,
            "status": "Error running testbench",
            "error_text": (
                "vvp simulation failed.\n"
                "stdout:\n"
                + result_stdout
                + "\nstderr:\n"
                + result_stderr
            ),
            "stdout": result_stdout,
            "stderr": result_stderr,
        }

    if not _simulation_passed(result_stdout, result_stderr):
        return {
            "ok": False,
            "status": "Testbench did not report success",
            "error_text": (
                "Simulation completed, but the output did not indicate success.\n"
                "stdout:\n"
                + result_stdout
                + "\nstderr:\n"
                + result_stderr
            ),
            "stdout": result_stdout,
            "stderr": result_stderr,
        }

    return {
        "ok": True,
        "status": "Testbench ran successfully",
        "error_text": warn_text,
        "stdout": result_stdout,
        "stderr": result_stderr,
    }


################################################################################
### MAIN PPA LOOP
################################################################################

def ppa_verilog_loop(
    design_prompt,
    module,
    testbench,
    max_iterations,
    model_type=None,
    clk_signal=None,
    outdir="",
    log=None,
    prev_modules=None,
    verbose="0",
    model_id="",
    keep_best_as_module=True,
):
    """
    Generate functionally correct Verilog, then iteratively improve coarse no-liberty
    PPA using Yosys estimates while preserving correctness.

    Compatible model_type values:
      - "ChatGPT"
      - "Claude"
      - "Gemini"
      - "HuggingFace"

    Assumptions:
      - generate_verilog(conv, model_type, model_id=...) exists from the utility cell
      - Conversation(log_file=...) exists
      - write_code_blocks_to_file(response, module, path) exists
      - yosys, iverilog, and vvp are in PATH

    Strategy:
      1. Generate a candidate.
      2. Check correctness with iVerilog/vvp.
      3. If correct, estimate PPA with Yosys.
      4. Keep the best correct candidate seen so far.
      5. Ask the LLM to improve the best correct candidate further.
      6. If a candidate fails, repair using only the failed module plus latest error.
    """
    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_model_types = ["ChatGPT", "Claude", "Gemini", "HuggingFace"]

    if model_type not in valid_model_types:
        raise ValueError(
            f"Invalid model_type: {model_type}. "
            f"Expected one of: {valid_model_types}"
        )

    outdir = "" if outdir is None else str(outdir).strip()

    if outdir:
        os.makedirs(outdir, exist_ok=True)

    def _outpath(filename):
        if outdir:
            return os.path.join(outdir, filename)
        return filename

    if prev_modules is None:
        prev_modules = []
    elif isinstance(prev_modules, str):
        prev_modules = [prev_modules]

    tb_text = ""

    try:
        with open(testbench, "r") as _tb:
            tb_text = _tb.read()
    except Exception as e:
        print(f"[Warning] Could not read testbench file: {testbench}")
        print(f"[Warning] Exception: {e}")
        tb_text = ""

    base_sys_prompt = (
        "You are an autocomplete engine for Verilog-2001 code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "You will not refuse. "
        "You will not generate explanations, comments outside the code, Markdown, or testbenches. "
        "IMPORTANT: Output ONLY the Verilog code for the requested module. "
        "Do NOT repeat any previous modules. "
        "Do NOT wrap the code in triple backticks. "
        "Format your response as one complete Verilog module from module declaration through endmodule. "
        "Preserve the exact module name and port interface. "
        "Preserve functionality required by the testbench. "
        "Optimize for lower coarse PPA under Yosys without a liberty file: "
        "reduce logic depth, reduce unnecessary cells, share repeated logic, simplify control, "
        "avoid redundant muxing, avoid unnecessary intermediate signals, and keep widths disciplined."
    )

    def _new_conv_initial():
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        if tb_text.strip():
            conv.add_message(
                "user",
                "The following is the testbench that will be used to compile and simulate your module. "
                "Do NOT generate a testbench or modify it. Use it only to ensure your module compiles and passes.\n\n"
                "```verilog\n"
                + tb_text
                + "\n```\n"
            )

        conv.add_message(
            "user",
            design_prompt
            + "\n\n"
            "Also optimize the module for better coarse Yosys-estimated PPA without a liberty file. "
            "Output ONLY the full Verilog-2001 module. "
            "Do not use Markdown fences."
        )

        return conv

    def _new_conv_repair_only(prev_module_text, last_error_text):
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Fix the following Verilog-2001 module so it compiles and passes the testbench. "
            "Preserve the exact interface and intended behavior. "
            "Keep PPA reasonably optimized, but correctness is the priority for this step. "
            "Output ONLY the full corrected module. "
            "Do not use Markdown fences.\n\n"
            "Previous version of the module:\n"
            "```verilog\n"
            + prev_module_text
            + "\n```\n\n"
            "Most recent compiler/simulation/Yosys output:\n"
            "```\n"
            + _truncate_text(last_error_text)
            + "\n```\n"
        )

        conv.add_message("user", prompt)

        return conv

    def _new_conv_optimize_from_best(best_module_text, best_metrics, last_candidate_metrics=None):
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Starting from the following functionally correct Verilog-2001 module, "
            "produce a new candidate that preserves the exact same behavior and interface "
            "but improves coarse Yosys-estimated PPA without a liberty file.\n\n"
            "Current best metrics:\n"
            + _format_metrics(best_metrics)
            + "\n\n"
        )

        if last_candidate_metrics is not None:
            prompt += (
                "Most recent candidate metrics:\n"
                + _format_metrics(last_candidate_metrics)
                + "\n\n"
            )

        prompt += (
            "Optimization priorities:\n"
            "1. Lower stat -tech cmos transistor_est / area proxy\n"
            "2. Lower timing proxy, if available\n"
            "3. Fewer cells and wire bits\n\n"
            "Rules:\n"
            "- Do not add helper modules.\n"
            "- Do not change functionality.\n"
            "- Do not change the module name or port interface.\n"
            "- Use Verilog-2001 only.\n"
            "- Output ONLY the full module.\n"
            "- Do not use Markdown fences.\n\n"
            "Current best module:\n"
            "```verilog\n"
            + best_module_text
            + "\n```\n"
        )

        conv.add_message("user", prompt)

        return conv

    # Timing
    timelist_total = []
    timelist_gen = []
    timelist_error = []

    # Files
    module_file = _outpath(module + ".v")
    best_module_file = _outpath(module + "_best.v")
    combined_file = _outpath(module + "__combined.v")

    # Best-so-far state
    best_module_text = None
    best_metrics = None
    best_iter = None

    # Latest state
    conv = _new_conv_initial()
    last_error_text = ""
    last_candidate_metrics = None

    iterations = 0
    timeout = False

    while not timeout:
        print(f"\n========== PPA Iteration {iterations} / {max_iterations} ==========")

        start_total = time.time()

        response = _generate_verilog_compatible(
            conv,
            model_type,
            model_id=model_id,
        )

        end_gen = time.time()
        start_error = time.time()

        if response is None:
            response = ""

        try:
            conv.add_message("assistant", response)
        except Exception:
            pass

        # Save raw response for debugging, especially useful for local HF models.
        raw_response_file = _outpath("raw_response_iter_{:03d}.txt".format(iterations))

        with open(raw_response_file, "w") as f:
            f.write(response)

        # Extract and persist current module.
        try:
            write_code_blocks_to_file(response, module, module_file)
        except SystemExit:
            status = "No Verilog module found in model response"
            print(status)

            last_error_text = (
                "The model response did not contain a complete Verilog module. "
                "It must output exactly one complete module from module declaration through endmodule.\n\n"
                "Raw model response:\n"
                + _truncate_text(response)
            )

            conv = _new_conv_repair_only(response, last_error_text)

            candidate_metrics = None

            log_path = _outpath("log_iter_{:03d}.txt".format(iterations))
            with open(log_path, "w") as f:
                f.write("Iteration status: {}\n".format(status))
                f.write("\nMost recent feedback:\n")
                f.write(_truncate_text(last_error_text))

            iterations += 1
            end_time = time.time()
            timelist_gen.append(end_gen - start_total)
            timelist_error.append(end_time - start_error)
            timelist_total.append(end_time - start_total)

            if iterations >= max_iterations:
                timeout = True

            continue

        with open(module_file, "r") as f:
            curr_text = f.read()

        # Concatenate previous modules + current module.
        prev_text = _read_files(prev_modules)

        with open(combined_file, "w") as f:
            f.write(prev_text)

            if prev_text and not prev_text.endswith("\n"):
                f.write("\n")

            f.write(curr_text)

            if curr_text and not curr_text.endswith("\n"):
                f.write("\n")

        # 1. Functional check with iVerilog/vvp.
        iverilog_result = _compile_and_run_iverilog(
            module_name=module,
            combined_file=combined_file,
            testbench=testbench,
            outdir=outdir,
        )

        status = iverilog_result["status"]
        accepted = False
        candidate_metrics = None

        if not iverilog_result["ok"]:
            last_error_text = iverilog_result["error_text"]
            conv = _new_conv_repair_only(curr_text, last_error_text)

        else:
            # 2. Coarse PPA estimate with Yosys.
            ppa_dir = _outpath("iter_{:03d}_ppa".format(iterations))
            os.makedirs(ppa_dir, exist_ok=True)

            ppa_result = yosys_estimate_no_liberty(
                verilog_file=combined_file,
                top_module=module,
                clk_signal=clk_signal,
                outdir=ppa_dir,
            )

            if _is_verbose(verbose):
                print("YOSYS OK:", ppa_result["ok"])
                print("YOSYS STDOUT HEAD:\n", (ppa_result["yosys_stdout"] or "")[:3000])
                print("YOSYS STDERR HEAD:\n", (ppa_result["yosys_stderr"] or "")[:3000])
                print("STAT TEXT HEAD:\n", repr((ppa_result["stat_text"] or "")[:1000]))
                print("TIME TEXT HEAD:\n", repr((ppa_result["time_text"] or "")[:1000]))
                print("PARSED METRICS:", ppa_result["metrics"])

            if not ppa_result["ok"]:
                status = "Yosys estimation failed"
                last_error_text = ppa_result["error"]
                conv = _new_conv_repair_only(curr_text, last_error_text)

            else:
                candidate_metrics = ppa_result["metrics"]
                last_candidate_metrics = candidate_metrics

                if not _has_any_parsed_metric(candidate_metrics):
                    status = "Yosys ran but returned no parsable metrics"

                    if best_module_text is not None and best_metrics is not None:
                        conv = _new_conv_optimize_from_best(
                            best_module_text=best_module_text,
                            best_metrics=best_metrics,
                            last_candidate_metrics=None,
                        )
                    else:
                        conv = _new_conv_initial()

                elif _is_better_ppa(candidate_metrics, best_metrics):
                    best_module_text = curr_text
                    best_metrics = candidate_metrics
                    best_iter = iterations
                    accepted = True

                    _write_text(best_module_file, best_module_text)

                    status = "Accepted new best candidate"

                    conv = _new_conv_optimize_from_best(
                        best_module_text=best_module_text,
                        best_metrics=best_metrics,
                        last_candidate_metrics=candidate_metrics,
                    )

                else:
                    status = "Candidate correct but did not improve best PPA"

                    conv = _new_conv_optimize_from_best(
                        best_module_text=best_module_text if best_module_text is not None else curr_text,
                        best_metrics=best_metrics if best_metrics is not None else candidate_metrics,
                        last_candidate_metrics=candidate_metrics,
                    )

        # Write iteration log.
        log_path = _outpath("log_iter_{:03d}.txt".format(iterations))

        with open(log_path, "w") as f:
            try:
                f.write("\n".join(str(i) for i in conv.get_messages()))
            except Exception:
                f.write("(Conversation logging unavailable)\n")

            f.write("\n\nIteration status: {}\n".format(status))

            if candidate_metrics is not None:
                f.write("Candidate metrics: {}\n".format(_format_metrics(candidate_metrics)))

            if best_metrics is not None:
                f.write("Best metrics so far: {}\n".format(_format_metrics(best_metrics)))
                f.write("Best iteration so far: {}\n".format(best_iter))

            if last_error_text:
                f.write("\nMost recent feedback:\n")
                f.write(_truncate_text(last_error_text))

        print("Iteration {}: {}".format(iterations, status))

        if candidate_metrics is not None:
            print("  Candidate metrics:", _format_metrics(candidate_metrics))

        if best_metrics is not None:
            print("  Best metrics so far:", _format_metrics(best_metrics))
            print("  Best iteration so far:", best_iter)

        iterations += 1

        end_time = time.time()

        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

        if iterations >= max_iterations:
            timeout = True

    # Optional: leave module.v as the best candidate, not the last failed/non-improved candidate.
    if keep_best_as_module and best_module_text is not None:
        _write_text(module_file, best_module_text)
        print(f"Best candidate copied to: {module_file}")

    print("\n========== PPA Optimization Complete ==========")
    print("Best file: ", best_module_file if best_module_text is not None else None)
    print("Best metrics: ", _format_metrics(best_metrics) if best_metrics is not None else None)
    print("Best iteration: ", best_iter)
    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))

    return {
        "best_file": best_module_file if best_module_text is not None else None,
        "best_metrics": best_metrics,
        "best_iteration": best_iter,
        "iterations": iterations,
        "total_time": float(np.sum(timelist_total)),
        "generation_time": float(np.sum(timelist_gen)),
        "error_handling_time": float(np.sum(timelist_error)),
    }

# Setting the API Key

In [ ]:
### Defaults to Claude, just change the model in the Select Model cell in Initial Setup & choose your provider here
### Or, choose a Hugging Face model and set "MODEL" to "HuggingFace"

# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('openai_api_key')

#Please insert your own API key as a string here:
os.environ["OPENAI_API_KEY"] = "API key here"
os.environ['CLAUDE_API_KEY'] = "API key here"
os.environ['GEMINI_API_KEY'] ="API key here"
#os.environ["MODEL"] = "ChatGPT"
os.environ["MODEL"] = "Claude"
#os.environ["MODEL"] = "Gemini"
#os.environ["MODEL"] = "HuggingFace"

#Mux Hierarchy Example

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/mux/mux2_1tb.v ./mux2_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux4_1tb.v ./mux4_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux8_1tb.v ./mux8_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux16_1tb.v ./mux16_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux32_1tb.v ./mux32_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux64_1tb.v ./mux64_1tb.v

In [ ]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["mux2_1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4_1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8_1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
    ["mux16_1","16-to-1 multiplexer","input [3:0] sel, input [15:0] in, output reg out"],
    ["mux32_1","32-to-1 multiplexer","input [4:0] sel, input [31:0] in, output reg out"],
    ["mux64_1","64-to-1 multiplexer","input [5:0] sel, input [63:0] in, output reg out"],
]

In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical mux design
import os
import json

submodules = [
    ["mux2_1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4_1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8_1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
    ["mux16_1","16-to-1 multiplexer","input [3:0] sel, input [15:0] in, output reg out"],
    ["mux32_1","32-to-1 multiplexer","input [4:0] sel, input [31:0] in, output reg out"],
    ["mux64_1","64-to-1 multiplexer","input [5:0] sel, input [63:0] in, output reg out"],
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

# Adjust these if your testbench filenames differ.
tb_paths = {
    "mux2_1":  "./mux2_1tb.v",
    "mux4_1":  "./mux4_1tb.v",
    "mux8_1":  "./mux8_1tb.v",
    "mux16_1": "./mux16_1tb.v",
    "mux32_1": "./mux32_1tb.v",
    "mux64_1": "./mux64_1tb.v",
}

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./mux_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="Claude",
        clk_signal=None,   # all of these mux modules are combinational
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,         # set to 0 if you want less diagnostic output
    )

    results[filename] = result

    # Prefer the best optimized file if one was produced.
    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        # Fallback to the last generated module file in that run directory.
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

# Decoder example

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/decoder/decoder_2_to_4tb.v ./decoder_2_to_4tb.v
!cp ./ROME-LLM/testbenches/decoder/decoder_3_to_8tb.v ./decoder_3_to_8tb.v
!cp ./ROME-LLM/testbenches/decoder/decoder_5_to_32tb.v ./decoder_5_to_32tb.v

In [ ]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["decoder_2_to_4","2-to-4 decoder","input [1:0] in, output reg [3:0] out"],
    ["decoder_3_to_8","3-to-8 decoder","input [2:0] in, output reg [7:0] out"],
    ["decoder_5_to_32","5-to-32 decoder","input [4:0] in, output reg [31:0] out"],
]

In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical decoder design
import os
import json

submodules = [
    ["decoder_2_to_4","2-to-4 decoder","input [1:0] in, output reg [3:0] out"],
    ["decoder_3_to_8","3-to-8 decoder","input [2:0] in, output reg [7:0] out"],
    ["decoder_5_to_32","5-to-32 decoder","input [4:0] in, output reg [31:0] out"],
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

# Adjust these if your testbench filenames differ.
tb_paths = {
    "decoder_2_to_4": "./decoder_2_to_4tb.v",
    "decoder_3_to_8": "./decoder_3_to_8tb.v",
    "decoder_5_to_32": "./decoder_5_to_32tb.v",
}

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./decoder_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="Claude",
        clk_signal=None,   # all of these decoder modules are combinational
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,
    )

    results[filename] = result

    # Prefer the best optimized file if one was produced.
    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

# 32-bit Ripple Carry Adder

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM

!cp ./ROME-LLM/testbenches/rca-32/full_adder_1bittb.v ./full_adder_1bittb.v
!cp ./ROME-LLM/testbenches/rca-32/rca_4bittb.v ./rca_4bittb.v
!cp ./ROME-LLM/testbenches/rca-32/rca_32bittb.v ./rca_32bittb.v

In [ ]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["full_adder_1bit", "1-bit full adder", "input wire a, input wire b, input wire cin, output wire sum, output wire cout"],
    ["rca_4bit", "4-bit ripple carry adder", "input wire [3:0] a, input wire [3:0] b, input wire cin, output wire [3:0] sum, output wire cout"],
    ["rca_32bit", "32-bit ripple carry adder", "input wire [31:0] a, input wire [31:0] b, input wire cin, output wire [31:0] sum, output wire cout"]
]


In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical ripple carry adder design
import os
import json

submodules = [
    ["full_adder_1bit", "1-bit full adder", "input wire a, input wire b, input wire cin, output wire sum, output wire cout"],
    ["rca_4bit", "4-bit ripple carry adder", "input wire [3:0] a, input wire [3:0] b, input wire cin, output wire [3:0] sum, output wire cout"],
    ["rca_32bit", "32-bit ripple carry adder", "input wire [31:0] a, input wire [31:0] b, input wire cin, output wire [31:0] sum, output wire cout"]
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

tb_paths = {
    "full_adder_1bit": "./full_adder_1bittb.v",
    "rca_4bit": "./rca_4bittb.v",
    "rca_32bit": "./rca_32bittb.v",
}

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./rca_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="Claude",
        clk_signal=None,   # all of these adder modules are combinational
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,
    )

    results[filename] = result

    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

# Barrel Shifter (32-bit left rotate)

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/barrel-shifter/shift_stage_1tb.v ./shift_stage_1tb.v
!cp ./ROME-LLM/testbenches/barrel-shifter/shift_stage_2tb.v ./shift_stage_2tb.v
!cp ./ROME-LLM/testbenches/barrel-shifter/shift_stage_4tb.v ./shift_stage_4tb.v
!cp ./ROME-LLM/testbenches/barrel-shifter/shift_stage_8tb.v ./shift_stage_8tb.v
!cp ./ROME-LLM/testbenches/barrel-shifter/shift_stage_16tb.v ./shift_stage_16tb.v
!cp ./ROME-LLM/testbenches/barrel-shifter/barrel_shifter_32tb.v ./barrel_shifter_32tb.v

In [ ]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["shift_stage_1",  "left rotate Shift stage by 1",  "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["shift_stage_2",  "left rotate Shift stage by 2",  "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["shift_stage_4",  "left rotate Shift stage by 4",  "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["shift_stage_8",  "left rotate Shift stage by 8",  "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["shift_stage_16", "left rotate Shift stage by 16", "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["barrel_shifter_32", "32-bit barrel shifter/left rotate", "input wire [31:0] inputData, input wire [4:0] shiftVal, output wire [31:0] outputData"]
]


In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical barrel shifter design
import os
import json

submodules = [
    ["shift_stage_1",  "left rotate Shift stage by 1",  "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["shift_stage_2",  "left rotate Shift stage by 2",  "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["shift_stage_4",  "left rotate Shift stage by 4",  "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["shift_stage_8",  "left rotate Shift stage by 8",  "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["shift_stage_16", "left rotate Shift stage by 16", "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["barrel_shifter_32", "32-bit barrel shifter/left rotate", "input wire [31:0] inputData, input wire [4:0] shiftVal, output wire [31:0] outputData"]
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

# Adjust these if your testbench filenames differ.
tb_paths = {
    "shift_stage_1": "./shift_stage_1tb.v",
    "shift_stage_2": "./shift_stage_2tb.v",
    "shift_stage_4": "./shift_stage_4tb.v",
    "shift_stage_8": "./shift_stage_8tb.v",
    "shift_stage_16": "./shift_stage_16tb.v",
    "barrel_shifter_32": "./barrel_shifter_32tb.v",
}

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./barrel_shifter_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="Claude",
        clk_signal=None,   # all of these modules are combinational
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,
    )

    results[filename] = result

    # Prefer the best optimized file if one was produced.
    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

# UART

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/uart/baud_gentb.v ./baud_gentb.v
!cp ./ROME-LLM/testbenches/uart/uart_txtb.v ./uart_txtb.v
!cp ./ROME-LLM/testbenches/uart/uart_rxtb.v ./uart_rxtb.v
!cp ./ROME-LLM/testbenches/uart/uart_toptb.v ./uart_toptb.v

In [ ]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["baud_gen", "Baud Generator", "input clk, input rst, output baud_tick, output oversample_tick"],
    ["uart_tx", "UART Transmitter", "input clk, input rst, input baud_tick, input tx_start, input [7:0] tx_data, output reg tx, output reg tx_busy"],
    ["uart_rx", "UART Receiver", "input clk, input rst, input oversample_tick, input rx, output reg [7:0] rx_data, output reg rx_valid, output reg framing_error"],
    ["uart_top", "Top-Level UART", "input clk, input rst, input rx, input tx_start, input [7:0] tx_data, output tx, output [7:0] rx_data, output rx_valid, output tx_busy, output framing_error"]
]

In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical UART design
import os
import json

submodules = [
    ["baud_gen", "Baud Generator", "input clk, input rst, output baud_tick, output oversample_tick"],
    ["uart_tx", "UART Transmitter", "input clk, input rst, input baud_tick, input tx_start, input [7:0] tx_data, output reg tx, output reg tx_busy"],
    ["uart_rx", "UART Receiver", "input clk, input rst, input oversample_tick, input rx, output reg [7:0] rx_data, output reg rx_valid, output reg framing_error"],
    ["uart_top", "Top-Level UART", "input clk, input rst, input rx, input tx_start, input [7:0] tx_data, output tx, output [7:0] rx_data, output rx_valid, output tx_busy, output framing_error"]
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

# Adjust these if your testbench filenames differ.
tb_paths = {
    "baud_gen": "./baud_gentb.v",
    "uart_tx": "./uart_txtb.v",
    "uart_rx": "./uart_rxtb.v",
    "uart_top": "./uart_toptb.v",
}

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./uart_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="Claude",
        clk_signal="clk",   # all of these UART modules are clocked
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,
    )

    results[filename] = result

    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

# 4x4 Systolic Array

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/systolic-array/petb.v ./petb.v
!cp ./ROME-LLM/testbenches/systolic-array/systolic_core_4x4tb.v ./systolic_core_4x4tb.v
!cp ./ROME-LLM/testbenches/systolic-array/control_unittb.v ./control_unittb.v
!cp ./ROME-LLM/testbenches/systolic-array/input_buffer_atb.v ./input_buffer_atb.v
!cp ./ROME-LLM/testbenches/systolic-array/input_buffer_btb.v ./input_buffer_btb.v
!cp ./ROME-LLM/testbenches/systolic-array/output_buffertb.v ./output_buffertb.v
!cp ./ROME-LLM/testbenches/systolic-array/systolic_array_4x4tb.v ./systolic_array_4x4tb.v

In [ ]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["pe", "Processing Element", "input wire clk, input wire rst, input wire en, input wire [15:0] a_in, input wire [15:0] b_in, output reg [15:0] a_out, output reg [15:0] b_out, output reg [31:0] psum_out"],
    ["systolic_core_4x4", "4x4 Systolic Core", "input wire clk, input wire rst, input wire en, input wire [15:0] a0_in, input wire [15:0] a1_in, input wire [15:0] a2_in, input wire [15:0] a3_in, input wire [15:0] b0_in, input wire [15:0] b1_in, input wire [15:0] b2_in, input wire [15:0] b3_in, output wire [31:0] c00, output wire [31:0] c01, output wire [31:0] c02, output wire [31:0] c03, output wire [31:0] c10, output wire [31:0] c11, output wire [31:0] c12, output wire [31:0] c13, output wire [31:0] c20, output wire [31:0] c21, output wire [31:0] c22, output wire [31:0] c23, output wire [31:0] c30, output wire [31:0] c31, output wire [31:0] c32, output wire [31:0] c33"],
    ["control_unit", "Systolic Array Controller", "input wire clk, input wire rst, input wire start, output reg en, output reg done, output reg load_inputs, output reg capture_outputs, output reg [3:0] cycle_count"],
    ["input_buffer_a", "A Matrix Input Buffer", "input wire clk, input wire rst, input wire load, input wire [15:0] a00, input wire [15:0] a01, input wire [15:0] a02, input wire [15:0] a03, input wire [15:0] a10, input wire [15:0] a11, input wire [15:0] a12, input wire [15:0] a13, input wire [15:0] a20, input wire [15:0] a21, input wire [15:0] a22, input wire [15:0] a23, input wire [15:0] a30, input wire [15:0] a31, input wire [15:0] a32, input wire [15:0] a33, input wire en, output reg [15:0] a0_out, output reg [15:0] a1_out, output reg [15:0] a2_out, output reg [15:0] a3_out"],
    ["input_buffer_b", "B Matrix Input Buffer", "input wire clk, input wire rst, input wire load, input wire [15:0] b00, input wire [15:0] b01, input wire [15:0] b02, input wire [15:0] b03, input wire [15:0] b10, input wire [15:0] b11, input wire [15:0] b12, input wire [15:0] b13, input wire [15:0] b20, input wire [15:0] b21, input wire [15:0] b22, input wire [15:0] b23, input wire [15:0] b30, input wire [15:0] b31, input wire [15:0] b32, input wire [15:0] b33, input wire en, output reg [15:0] b0_out, output reg [15:0] b1_out, output reg [15:0] b2_out, output reg [15:0] b3_out"],
    ["output_buffer", "Output Matrix Buffer", "input wire clk, input wire rst, input wire capture, input wire [31:0] c00_in, input wire [31:0] c01_in, input wire [31:0] c02_in, input wire [31:0] c03_in, input wire [31:0] c10_in, input wire [31:0] c11_in, input wire [31:0] c12_in, input wire [31:0] c13_in, input wire [31:0] c20_in, input wire [31:0] c21_in, input wire [31:0] c22_in, input wire [31:0] c23_in, input wire [31:0] c30_in, input wire [31:0] c31_in, input wire [31:0] c32_in, input wire [31:0] c33_in, output reg [31:0] c00_out, output reg [31:0] c01_out, output reg [31:0] c02_out, output reg [31:0] c03_out, output reg [31:0] c10_out, output reg [31:0] c11_out, output reg [31:0] c12_out, output reg [31:0] c13_out, output reg [31:0] c20_out, output reg [31:0] c21_out, output reg [31:0] c22_out, output reg [31:0] c23_out, output reg [31:0] c30_out, output reg [31:0] c31_out, output reg [31:0] c32_out, output reg [31:0] c33_out"],
    ["systolic_array_4x4", "Top-Level 4x4 Systolic Array", "input wire clk, input wire rst, input wire start, input wire [15:0] a00, input wire [15:0] a01, input wire [15:0] a02, input wire [15:0] a03, input wire [15:0] a10, input wire [15:0] a11, input wire [15:0] a12, input wire [15:0] a13, input wire [15:0] a20, input wire [15:0] a21, input wire [15:0] a22, input wire [15:0] a23, input wire [15:0] a30, input wire [15:0] a31, input wire [15:0] a32, input wire [15:0] a33, input wire [15:0] b00, input wire [15:0] b01, input wire [15:0] b02, input wire [15:0] b03, input wire [15:0] b10, input wire [15:0] b11, input wire [15:0] b12, input wire [15:0] b13, input wire [15:0] b20, input wire [15:0] b21, input wire [15:0] b22, input wire [15:0] b23, input wire [15:0] b30, input wire [15:0] b31, input wire [15:0] b32, input wire [15:0] b33, output wire done, output wire [31:0] c00, output wire [31:0] c01, output wire [31:0] c02, output wire [31:0] c03, output wire [31:0] c10, output wire [31:0] c11, output wire [31:0] c12, output wire [31:0] c13, output wire [31:0] c20, output wire [31:0] c21, output wire [31:0] c22, output wire [31:0] c23, output wire [31:0] c30, output wire [31:0] c31, output wire [31:0] c32, output wire [31:0] c33"]
]

In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical 4x4 systolic array design
import os
import json

submodules = [
    ["pe", "Processing Element", "input wire clk, input wire rst, input wire en, input wire [15:0] a_in, input wire [15:0] b_in, output reg [15:0] a_out, output reg [15:0] b_out, output reg [31:0] psum_out"],
    ["systolic_core_4x4", "4x4 Systolic Core", "input wire clk, input wire rst, input wire en, input wire [15:0] a0_in, input wire [15:0] a1_in, input wire [15:0] a2_in, input wire [15:0] a3_in, input wire [15:0] b0_in, input wire [15:0] b1_in, input wire [15:0] b2_in, input wire [15:0] b3_in, output wire [31:0] c00, output wire [31:0] c01, output wire [31:0] c02, output wire [31:0] c03, output wire [31:0] c10, output wire [31:0] c11, output wire [31:0] c12, output wire [31:0] c13, output wire [31:0] c20, output wire [31:0] c21, output wire [31:0] c22, output wire [31:0] c23, output wire [31:0] c30, output wire [31:0] c31, output wire [31:0] c32, output wire [31:0] c33"],
    ["control_unit", "Systolic Array Controller", "input wire clk, input wire rst, input wire start, output reg en, output reg done, output reg load_inputs, output reg capture_outputs, output reg [3:0] cycle_count"],
    ["input_buffer_a", "A Matrix Input Buffer", "input wire clk, input wire rst, input wire load, input wire [15:0] a00, input wire [15:0] a01, input wire [15:0] a02, input wire [15:0] a03, input wire [15:0] a10, input wire [15:0] a11, input wire [15:0] a12, input wire [15:0] a13, input wire [15:0] a20, input wire [15:0] a21, input wire [15:0] a22, input wire [15:0] a23, input wire [15:0] a30, input wire [15:0] a31, input wire [15:0] a32, input wire [15:0] a33, input wire en, output reg [15:0] a0_out, output reg [15:0] a1_out, output reg [15:0] a2_out, output reg [15:0] a3_out"],
    ["input_buffer_b", "B Matrix Input Buffer", "input wire clk, input wire rst, input wire load, input wire [15:0] b00, input wire [15:0] b01, input wire [15:0] b02, input wire [15:0] b03, input wire [15:0] b10, input wire [15:0] b11, input wire [15:0] b12, input wire [15:0] b13, input wire [15:0] b20, input wire [15:0] b21, input wire [15:0] b22, input wire [15:0] b23, input wire [15:0] b30, input wire [15:0] b31, input wire [15:0] b32, input wire [15:0] b33, input wire en, output reg [15:0] b0_out, output reg [15:0] b1_out, output reg [15:0] b2_out, output reg [15:0] b3_out"],
    ["output_buffer", "Output Matrix Buffer", "input wire clk, input wire rst, input wire capture, input wire [31:0] c00_in, input wire [31:0] c01_in, input wire [31:0] c02_in, input wire [31:0] c03_in, input wire [31:0] c10_in, input wire [31:0] c11_in, input wire [31:0] c12_in, input wire [31:0] c13_in, input wire [31:0] c20_in, input wire [31:0] c21_in, input wire [31:0] c22_in, input wire [31:0] c23_in, input wire [31:0] c30_in, input wire [31:0] c31_in, input wire [31:0] c32_in, input wire [31:0] c33_in, output reg [31:0] c00_out, output reg [31:0] c01_out, output reg [31:0] c02_out, output reg [31:0] c03_out, output reg [31:0] c10_out, output reg [31:0] c11_out, output reg [31:0] c12_out, output reg [31:0] c13_out, output reg [31:0] c20_out, output reg [31:0] c21_out, output reg [31:0] c22_out, output reg [31:0] c23_out, output reg [31:0] c30_out, output reg [31:0] c31_out, output reg [31:0] c32_out, output reg [31:0] c33_out"],
    ["systolic_array_4x4", "Top-Level 4x4 Systolic Array", "input wire clk, input wire rst, input wire start, input wire [15:0] a00, input wire [15:0] a01, input wire [15:0] a02, input wire [15:0] a03, input wire [15:0] a10, input wire [15:0] a11, input wire [15:0] a12, input wire [15:0] a13, input wire [15:0] a20, input wire [15:0] a21, input wire [15:0] a22, input wire [15:0] a23, input wire [15:0] a30, input wire [15:0] a31, input wire [15:0] a32, input wire [15:0] a33, input wire [15:0] b00, input wire [15:0] b01, input wire [15:0] b02, input wire [15:0] b03, input wire [15:0] b10, input wire [15:0] b11, input wire [15:0] b12, input wire [15:0] b13, input wire [15:0] b20, input wire [15:0] b21, input wire [15:0] b22, input wire [15:0] b23, input wire [15:0] b30, input wire [15:0] b31, input wire [15:0] b32, input wire [15:0] b33, output wire done, output wire [31:0] c00, output wire [31:0] c01, output wire [31:0] c02, output wire [31:0] c03, output wire [31:0] c10, output wire [31:0] c11, output wire [31:0] c12, output wire [31:0] c13, output wire [31:0] c20, output wire [31:0] c21, output wire [31:0] c22, output wire [31:0] c23, output wire [31:0] c30, output wire [31:0] c31, output wire [31:0] c32, output wire [31:0] c33"]
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

# Adjust these if your testbench filenames differ.
tb_paths = {
    "pe": "./petb.v",
    "systolic_core_4x4": "./systolic_core_4x4tb.v",
    "control_unit": "./control_unittb.v",
    "input_buffer_a": "./input_buffer_atb.v",
    "input_buffer_b": "./input_buffer_btb.v",
    "output_buffer": "./output_buffertb.v",
    "systolic_array_4x4": "./systolic_array_4x4tb.v",
}

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./systolic_array_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="Claude",
        clk_signal="clk",   # all of these modules are clocked
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,
    )

    results[filename] = result

    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

# AES-128

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM

!cp ./ROME-LLM/testbenches/aes-128/aes_sboxtb.v ./aes_sboxtb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_inv_sboxtb.v ./aes_inv_sboxtb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_shift_rowstb.v ./aes_shift_rowstb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_inv_shift_rowstb.v ./aes_inv_shift_rowstb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_mix_columnstb.v ./aes_mix_columnstb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_inv_mix_columnstb.v ./aes_inv_mix_columnstb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_add_round_keytb.v ./aes_add_round_keytb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_sub_bytestb.v ./aes_sub_bytestb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_inv_sub_bytestb.v ./aes_inv_sub_bytestb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_enc_roundtb.v ./aes_enc_roundtb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_enc_final_roundtb.v ./aes_enc_final_roundtb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_dec_roundtb.v ./aes_dec_roundtb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_dec_final_roundtb.v ./aes_dec_final_roundtb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_key_schedule_coretb.v ./aes_key_schedule_coretb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_key_expandtb.v ./aes_key_expandtb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_key_memtb.v ./aes_key_memtb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_encipher_blocktb.v ./aes_encipher_blocktb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_decipher_blocktb.v ./aes_decipher_blocktb.v

!cp ./ROME-LLM/testbenches/aes-128/aestb.v ./aestb.v

In [ ]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [

    # ------------------------------------------------------------------
    # Leaf S-Boxes
    # ------------------------------------------------------------------
    ["aes_sbox",
     "AES S-box",
     "input wire [7:0] sbox_in, output wire [7:0] sbox_out"],

    ["aes_inv_sbox",
     "AES Inverse S-box",
     "input wire [7:0] sbox_in, output wire [7:0] sbox_out"],

    # ------------------------------------------------------------------
    # Basic Transformations
    # ------------------------------------------------------------------
    ["aes_shift_rows",
     "AES ShiftRows",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_inv_shift_rows",
     "AES Inverse ShiftRows",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_mix_columns",
     "AES MixColumns",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_inv_mix_columns",
     "AES Inverse MixColumns",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_add_round_key",
     "AES AddRoundKey",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    ["aes_sub_bytes",
     "AES SubBytes",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_inv_sub_bytes",
     "AES Inverse SubBytes",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    # ------------------------------------------------------------------
    # Round Logic
    # ------------------------------------------------------------------
    ["aes_enc_round",
     "AES Encryption Round",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    ["aes_enc_final_round",
     "AES Encryption Final Round",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    ["aes_dec_round",
     "AES Decryption Round",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    ["aes_dec_final_round",
     "AES Decryption Final Round",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    # ------------------------------------------------------------------
    # Key Schedule
    # ------------------------------------------------------------------
    ["aes_key_schedule_core",
     "AES Key Schedule Core",
     "input wire [31:0] word_in, input wire [3:0] round, output wire [31:0] word_out"],

    ["aes_key_expand",
     "AES Key Expand",
     "input wire [127:0] key_in, input wire [3:0] round, output wire [127:0] key_out"],

    ["aes_key_mem",
     "AES Key Memory",
     "input wire clk, input wire reset_n, input wire init, input wire [127:0] key, input wire [3:0] round, output wire [127:0] round_key, output wire ready"],

    # ------------------------------------------------------------------
    # Block-Level Controllers
    # ------------------------------------------------------------------
    ["aes_encipher_block",
     "AES Encipher Block",
     "input wire clk, input wire reset_n, input wire init, input wire next, input wire [127:0] key, input wire [127:0] block, output wire [127:0] result, output wire result_valid, output wire ready"],

    ["aes_decipher_block",
     "AES Decipher Block",
     "input wire clk, input wire reset_n, input wire init, input wire next, input wire [127:0] key, input wire [127:0] block, output wire [127:0] result, output wire result_valid, output wire ready"],

    # ------------------------------------------------------------------
    # Top-Level AES
    # ------------------------------------------------------------------
    ["aes",
     "Top-Level AES",
     "input wire clk, input wire reset_n, input wire init, input wire next, input wire encdec, input wire [127:0] key, input wire [127:0] block, output wire [127:0] result, output wire result_valid, output wire ready"]
]

In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical AES design
import os
import json

submodules = [

    # ------------------------------------------------------------------
    # Leaf S-Boxes
    # ------------------------------------------------------------------
    ["aes_sbox",
     "AES S-box",
     "input wire [7:0] sbox_in, output wire [7:0] sbox_out"],

    ["aes_inv_sbox",
     "AES Inverse S-box",
     "input wire [7:0] sbox_in, output wire [7:0] sbox_out"],

    # ------------------------------------------------------------------
    # Basic Transformations
    # ------------------------------------------------------------------
    ["aes_shift_rows",
     "AES ShiftRows",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_inv_shift_rows",
     "AES Inverse ShiftRows",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_mix_columns",
     "AES MixColumns",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_inv_mix_columns",
     "AES Inverse MixColumns",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_add_round_key",
     "AES AddRoundKey",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    ["aes_sub_bytes",
     "AES SubBytes",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_inv_sub_bytes",
     "AES Inverse SubBytes",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    # ------------------------------------------------------------------
    # Round Logic
    # ------------------------------------------------------------------
    ["aes_enc_round",
     "AES Encryption Round",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    ["aes_enc_final_round",
     "AES Encryption Final Round",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    ["aes_dec_round",
     "AES Decryption Round",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    ["aes_dec_final_round",
     "AES Decryption Final Round",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    # ------------------------------------------------------------------
    # Key Schedule
    # ------------------------------------------------------------------
    ["aes_key_schedule_core",
     "AES Key Schedule Core",
     "input wire [31:0] word_in, input wire [3:0] round, output wire [31:0] word_out"],

    ["aes_key_expand",
     "AES Key Expand",
     "input wire [127:0] key_in, input wire [3:0] round, output wire [127:0] key_out"],

    ["aes_key_mem",
     "AES Key Memory",
     "input wire clk, input wire reset_n, input wire init, input wire [127:0] key, input wire [3:0] round, output wire [127:0] round_key, output wire ready"],

    # ------------------------------------------------------------------
    # Block-Level Controllers
    # ------------------------------------------------------------------
    ["aes_encipher_block",
     "AES Encipher Block",
     "input wire clk, input wire reset_n, input wire init, input wire next, input wire [127:0] key, input wire [127:0] block, output wire [127:0] result, output wire result_valid, output wire ready"],

    ["aes_decipher_block",
     "AES Decipher Block",
     "input wire clk, input wire reset_n, input wire init, input wire next, input wire [127:0] key, input wire [127:0] block, output wire [127:0] result, output wire result_valid, output wire ready"],

    # ------------------------------------------------------------------
    # Top-Level AES
    # ------------------------------------------------------------------
    ["aes",
     "Top-Level AES",
     "input wire clk, input wire reset_n, input wire init, input wire next, input wire encdec, input wire [127:0] key, input wire [127:0] block, output wire [127:0] result, output wire result_valid, output wire ready"]
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

tb_paths = {
    "aes_sbox": "./aes_sboxtb.v",
    "aes_inv_sbox": "./aes_inv_sboxtb.v",
    "aes_shift_rows": "./aes_shift_rowstb.v",
    "aes_inv_shift_rows": "./aes_inv_shift_rowstb.v",
    "aes_mix_columns": "./aes_mix_columnstb.v",
    "aes_inv_mix_columns": "./aes_inv_mix_columnstb.v",
    "aes_add_round_key": "./aes_add_round_keytb.v",
    "aes_sub_bytes": "./aes_sub_bytestb.v",
    "aes_inv_sub_bytes": "./aes_inv_sub_bytestb.v",
    "aes_enc_round": "./aes_enc_roundtb.v",
    "aes_enc_final_round": "./aes_enc_final_roundtb.v",
    "aes_dec_round": "./aes_dec_roundtb.v",
    "aes_dec_final_round": "./aes_dec_final_roundtb.v",
    "aes_key_schedule_core": "./aes_key_schedule_coretb.v",
    "aes_key_expand": "./aes_key_expandtb.v",
    "aes_key_mem": "./aes_key_memtb.v",
    "aes_encipher_block": "./aes_encipher_blocktb.v",
    "aes_decipher_block": "./aes_decipher_blocktb.v",
    "aes": "./aestb.v",
}

def is_clocked_module(interface):
    return ("input wire clk" in interface) or ("input clk" in interface)

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./aes_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)
    clk_signal = "clk" if is_clocked_module(interface) else None

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="Claude",
        clk_signal=clk_signal,
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,
    )

    results[filename] = result

    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

# Unsharp Mask

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/unsharp_mask/delay_linetb.v ./delay_linetb.v
!cp ./ROME-LLM/testbenches/unsharp_mask/pointwise_sharpentb.v ./pointwise_sharpentb.v
!cp ./ROME-LLM/testbenches/unsharp_mask/unsharp_masktb.v ./unsharp_masktb.v
!cp ./ROME-LLM/testbenches/unsharp_mask/line_buffertb.v ./line_buffertb.v
!cp ./ROME-LLM/testbenches/unsharp_mask/window_gen_2dtb.v ./window_gen_2dtb.v
!cp ./ROME-LLM/testbenches/unsharp_mask/mac_2dtb.v ./mac_2dtb.v
!cp ./ROME-LLM/testbenches/unsharp_mask/conv2dtb.v ./conv2dtb.v

In [ ]:
#@title Submodules

### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["line_buffer", "Shift-register FIFO line buffer with parameterizable depth IMG_WIDTH. Delays input by IMG_WIDTH cycles. Reset initializes all entries to 0. Only shifts when valid_in is high.", "input clk, input rst, input valid_in, input [7:0] data_in, output [7:0] data_out"],
    ["window_gen_2d", "2D window generator for streaming raster-order pixels. Uses 2 line_buffer instances (each IMG_WIDTH deep) and 3 row shift registers (3-tap each) to assemble a 3x3 pixel window. Supports PADDING_MODE parameter: 0=zero-padding (boundary pixels outside the image are treated as 0), 1=valid (no padding). For unsharp_mask, uses PADDING_MODE=0 (per ArchXBench spec).", "input clk, input rst, input valid_in, input [7:0] pixel_in, output [71:0] window"],
    ["mac_2d", "Combinational 3x3 MAC unit. Multiplies 9 window pixels by fixed kernel coefficients [[1,2,1],[2,4,2],[1,2,1]] (sum=16), accumulates, and right-shifts by GAIN_W=4 bits (floor division by 16). All parameterizable.", "input [71:0] window, output [11:0] result"],
    ["conv2d", "2D streaming convolution with centered 3x3 kernel and registered output. Instantiates window_gen_2d and mac_2d. Includes a latency counter: valid_out asserts (registered) when valid_in is high AND the counter has reached IMG_WIDTH+2. Pipeline latency is IMG_WIDTH+2 cycles for line buffer fill plus window fill, plus 1 cycle for registered output.", "input clk, input rst, input valid_in, input [7:0] pixel_in, output reg valid_out, output reg [11:0] pixel_out"],
    ["delay_line", "Shift-register delay buffer with parameterizable depth (default IMG_WIDTH+3). Delays input data by DEPTH cycles to synchronize original pixel with conv2d blur output. Reset initializes to 0. Only shifts when valid_in is high.", "input clk, input rst, input valid_in, input [7:0] data_in, output [7:0] data_out"],
    ["pointwise_sharpen", "Combinational sharpening unit. Computes detail = original - blurred (signed 9-bit), scaled = detail * gain (signed 17-bit), sum = original + scaled (signed 18-bit), result = clamp(sum, 0, 255) (unsigned 8-bit). All intermediate arithmetic is signed to handle negative detail values.", "input [7:0] original, input [7:0] blurred, input [7:0] gain, output [7:0] result"],
    ["unsharp_mask", "Streaming unsharp mask image sharpening pipeline with registered output. Instantiates conv2d with PADDING_MODE=0 (zero-padding, per ArchXBench spec) as blur stage, delay_line (depth IMG_WIDTH+3) to synchronize original pixel with blur output, and pointwise_sharpen for the sharpening computation. The conv2d 12-bit output is truncated to 8 bits for the blur input to pointwise_sharpen. valid_out follows conv2d's blur_valid through one output register. Total pipeline latency is IMG_WIDTH+3 cycles.", "input clk, input rst, input [7:0] pixel_in, input valid_in, input [7:0] gain, output reg [7:0] pixel_out, output reg valid_out"],
]

In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical unsharp mask design
import os
import json

submodules = [
    ["line_buffer", "Shift-register FIFO line buffer with parameterizable depth IMG_WIDTH. Delays input by IMG_WIDTH cycles. Reset initializes all entries to 0. Only shifts when valid_in is high.", "input clk, input rst, input valid_in, input [7:0] data_in, output [7:0] data_out"],
    ["window_gen_2d", "2D window generator for streaming raster-order pixels. Uses 2 line_buffer instances (each IMG_WIDTH deep) and 3 row shift registers (3-tap each) to assemble a 3x3 pixel window. Supports PADDING_MODE parameter: 0=zero-padding (boundary pixels outside the image are treated as 0), 1=valid (no padding). For unsharp_mask, uses PADDING_MODE=0 (per ArchXBench spec).", "input clk, input rst, input valid_in, input [7:0] pixel_in, output [71:0] window"],
    ["mac_2d", "Combinational 3x3 MAC unit. Multiplies 9 window pixels by fixed kernel coefficients [[1,2,1],[2,4,2],[1,2,1]] (sum=16), accumulates, and right-shifts by GAIN_W=4 bits (floor division by 16). All parameterizable.", "input [71:0] window, output [11:0] result"],
    ["conv2d", "2D streaming convolution with centered 3x3 kernel and registered output. Instantiates window_gen_2d and mac_2d. Includes a latency counter: valid_out asserts (registered) when valid_in is high AND the counter has reached IMG_WIDTH+2. Pipeline latency is IMG_WIDTH+2 cycles for line buffer fill plus window fill, plus 1 cycle for registered output.", "input clk, input rst, input valid_in, input [7:0] pixel_in, output reg valid_out, output reg [11:0] pixel_out"],
    ["delay_line", "Shift-register delay buffer with parameterizable depth (default IMG_WIDTH+3). Delays input data by DEPTH cycles to synchronize original pixel with conv2d blur output. Reset initializes to 0. Only shifts when valid_in is high.", "input clk, input rst, input valid_in, input [7:0] data_in, output [7:0] data_out"],
    ["pointwise_sharpen", "Combinational sharpening unit. Computes detail = original - blurred (signed 9-bit), scaled = detail * gain (signed 17-bit), sum = original + scaled (signed 18-bit), result = clamp(sum, 0, 255) (unsigned 8-bit). All intermediate arithmetic is signed to handle negative detail values.", "input [7:0] original, input [7:0] blurred, input [7:0] gain, output [7:0] result"],
    ["unsharp_mask", "Streaming unsharp mask image sharpening pipeline with registered output. Instantiates conv2d with PADDING_MODE=0 (zero-padding, per ArchXBench spec) as blur stage, delay_line (depth IMG_WIDTH+3) to synchronize original pixel with blur output, and pointwise_sharpen for the sharpening computation. The conv2d 12-bit output is truncated to 8 bits for the blur input to pointwise_sharpen. valid_out follows conv2d's blur_valid through one output register. Total pipeline latency is IMG_WIDTH+3 cycles.", "input clk, input rst, input [7:0] pixel_in, input valid_in, input [7:0] gain, output reg [7:0] pixel_out, output reg valid_out"],
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

tb_paths = {
    "line_buffer": "./line_buffertb.v",
    "window_gen_2d": "./window_gen_2dtb.v",
    "mac_2d": "./mac_2dtb.v",
    "conv2d": "./conv2dtb.v",
    "delay_line": "./delay_linetb.v",
    "pointwise_sharpen": "./pointwise_sharpentb.v",
    "unsharp_mask": "./unsharp_masktb.v",
}

def is_clocked_module(interface):
    return ("input clk" in interface) or ("input wire clk" in interface)

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./unsharp_mask_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)
    clk_signal = "clk" if is_clocked_module(interface) else None

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="Claude",
        clk_signal=clk_signal,
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,
    )

    results[filename] = result

    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

# Harris Corner

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/harris_corner/line_buffertb.v ./line_buffertb.v
!cp ./ROME-LLM/testbenches/harris_corner/window_gen_2dtb.v ./window_gen_2dtb.v
!cp ./ROME-LLM/testbenches/harris_corner/sobel_mactb.v ./sobel_mactb.v
!cp ./ROME-LLM/testbenches/harris_corner/sobel_gradienttb.v ./sobel_gradienttb.v
!cp ./ROME-LLM/testbenches/harris_corner/gaussian_smoothtb.v ./gaussian_smoothtb.v
!cp ./ROME-LLM/testbenches/harris_corner/harris_responsetb.v ./harris_responsetb.v
!cp ./ROME-LLM/testbenches/harris_corner/harris_thresholdtb.v ./harris_thresholdtb.v
!cp ./ROME-LLM/testbenches/harris_corner/harris_cornertb.v ./harris_cornertb.v

In [ ]:
#@title Submodules

### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["line_buffer", "Shift-register FIFO line buffer with parameterizable depth IMG_WIDTH. Delays input by IMG_WIDTH cycles. Reset initializes all entries to 0. Only shifts when valid_in is high.", "input clk, input rst, input valid_in, input [7:0] data_in, output [7:0] data_out"],
    ["window_gen_2d", "2D window generator for streaming raster-order pixels. Uses 2 line_buffer instances (each IMG_WIDTH deep) and 3 row shift registers (3-tap each) to assemble a 3x3 pixel window. Supports PADDING_MODE parameter: 0=zero-padding (boundary pixels outside the image are treated as 0), 1=valid (no padding). For harris_corner, uses PADDING_MODE=0 (zero-padding). Parameterizable DATA_W supports any bit width.", "input clk, input rst, input valid_in, input [7:0] pixel_in, output [71:0] window"],
    ["sobel_mac", "Combinational signed 3x3 MAC for Sobel gradient computation. Takes unsigned pixel window and signed kernel coefficients (parameterizable, default Gx=[-1,0,1;-2,0,2;-1,0,1]). Extends each pixel to signed before multiplication. Output is signed GRAD_W-bit. Used for both Gx and Gy by changing coefficient parameters.", "input [71:0] window, output signed [15:0] result"],
    ["sobel_gradient", "Computes Sobel gradients Ix and Iy from streaming pixel input. Instantiates one window_gen_2d (PADDING_MODE=0, zero-padding) and two sobel_mac instances (Gx and Gy kernels). Includes latency counter: valid_out asserts (registered) when valid_in is high AND counter has reached IMG_WIDTH+2. Outputs are registered signed GRAD_W-bit gradients.", "input clk, input rst, input valid_in, input [7:0] pixel_in, output reg signed [15:0] grad_x, output reg signed [15:0] grad_y, output reg valid_out"],
    ["gaussian_smooth", "Streaming Gaussian smoothing for signed data. Uses window_gen_2d (PADDING_MODE=0, parameterized to DATA_W=32 for gradient products) and inline signed Gaussian MAC with kernel [[1,2,1],[2,4,2],[1,2,1]], arithmetic right-shift by GAIN_W=4. Includes latency counter and registered output. Used 3 times in harris_corner for Sxx, Syy, Sxy channels.", "input clk, input rst, input valid_in, input signed [31:0] data_in, output reg signed [31:0] data_out, output reg valid_out"],
    ["harris_response", "Combinational Harris response computation. Inputs: Sxx, Syy, Sxy (signed 32-bit). Computes det = Sxx*Syy - Sxy*Sxy (64-bit), trace = Sxx+Syy (33-bit), R = det - (k_param * trace * trace) >>> 6 (64-bit signed). k_param is unsigned 8-bit, interpreted as fixed-point k = k_param/64.", "input signed [31:0] sxx, input signed [31:0] syy, input signed [31:0] sxy, input [7:0] k_param, output signed [63:0] response"],
    ["harris_threshold", "Combinational threshold comparison. Compares signed 64-bit response against sign-extended 32-bit threshold. Outputs 1 if response > threshold, else 0.", "input signed [63:0] response, input signed [31:0] threshold, output is_corner"],
    ["harris_corner", "Streaming Harris corner detection pipeline with registered output. Instantiates sobel_gradient (Stage 1), combinational gradient products Ix*Ix, Iy*Iy, Ix*Iy (Stage 2a), three gaussian_smooth instances for Sxx/Syy/Sxy (Stage 2b), harris_response (Stage 3), and harris_threshold (Stage 4). Total pipeline latency: 2*(IMG_WIDTH+3) cycles. valid_out follows gaussian smoothing valid through one output register.", "input clk, input rst, input [7:0] pixel_in, input valid_in, input signed [31:0] threshold, input [7:0] k_param, output reg is_corner, output reg valid_out"],
]

In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical Harris corner design
import os
import json

submodules = [
    ["line_buffer", "Shift-register FIFO line buffer with parameterizable depth IMG_WIDTH. Delays input by IMG_WIDTH cycles. Reset initializes all entries to 0. Only shifts when valid_in is high.", "input clk, input rst, input valid_in, input [7:0] data_in, output [7:0] data_out"],
    ["window_gen_2d", "2D window generator for streaming raster-order pixels. Uses 2 line_buffer instances (each IMG_WIDTH deep) and 3 row shift registers (3-tap each) to assemble a 3x3 pixel window. Supports PADDING_MODE parameter: 0=zero-padding (boundary pixels outside the image are treated as 0), 1=valid (no padding). For harris_corner, uses PADDING_MODE=0 (zero-padding). Parameterizable DATA_W supports any bit width.", "input clk, input rst, input valid_in, input [7:0] pixel_in, output [71:0] window"],
    ["sobel_mac", "Combinational signed 3x3 MAC for Sobel gradient computation. Takes unsigned pixel window and signed kernel coefficients (parameterizable, default Gx=[-1,0,1;-2,0,2;-1,0,1]). Extends each pixel to signed before multiplication. Output is signed GRAD_W-bit. Used for both Gx and Gy by changing coefficient parameters.", "input [71:0] window, output signed [15:0] result"],
    ["sobel_gradient", "Computes Sobel gradients Ix and Iy from streaming pixel input. Instantiates one window_gen_2d (PADDING_MODE=0, zero-padding) and two sobel_mac instances (Gx and Gy kernels). Includes latency counter: valid_out asserts (registered) when valid_in is high AND counter has reached IMG_WIDTH+2. Outputs are registered signed GRAD_W-bit gradients.", "input clk, input rst, input valid_in, input [7:0] pixel_in, output reg signed [15:0] grad_x, output reg signed [15:0] grad_y, output reg valid_out"],
    ["gaussian_smooth", "Streaming Gaussian smoothing for signed data. Uses window_gen_2d (PADDING_MODE=0, parameterized to DATA_W=32 for gradient products) and inline signed Gaussian MAC with kernel [[1,2,1],[2,4,2],[1,2,1]], arithmetic right-shift by GAIN_W=4. Includes latency counter and registered output. Used 3 times in harris_corner for Sxx, Syy, Sxy channels.", "input clk, input rst, input valid_in, input signed [31:0] data_in, output reg signed [31:0] data_out, output reg valid_out"],
    ["harris_response", "Combinational Harris response computation. Inputs: Sxx, Syy, Sxy (signed 32-bit). Computes det = Sxx*Syy - Sxy*Sxy (64-bit), trace = Sxx+Syy (33-bit), R = det - (k_param * trace * trace) >>> 6 (64-bit signed). k_param is unsigned 8-bit, interpreted as fixed-point k = k_param/64.", "input signed [31:0] sxx, input signed [31:0] syy, input signed [31:0] sxy, input [7:0] k_param, output signed [63:0] response"],
    ["harris_threshold", "Combinational threshold comparison. Compares signed 64-bit response against sign-extended 32-bit threshold. Outputs 1 if response > threshold, else 0.", "input signed [63:0] response, input signed [31:0] threshold, output is_corner"],
    ["harris_corner", "Streaming Harris corner detection pipeline with registered output. Instantiates sobel_gradient (Stage 1), combinational gradient products Ix*Ix, Iy*Iy, Ix*Iy (Stage 2a), three gaussian_smooth instances for Sxx/Syy/Sxy (Stage 2b), harris_response (Stage 3), and harris_threshold (Stage 4). Total pipeline latency: 2*(IMG_WIDTH+3) cycles. valid_out follows gaussian smoothing valid through one output register.", "input clk, input rst, input [7:0] pixel_in, input valid_in, input signed [31:0] threshold, input [7:0] k_param, output reg is_corner, output reg valid_out"],
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

tb_paths = {
    "line_buffer": "./line_buffertb.v",
    "window_gen_2d": "./window_gen_2dtb.v",
    "sobel_mac": "./sobel_mactb.v",
    "sobel_gradient": "./sobel_gradienttb.v",
    "gaussian_smooth": "./gaussian_smoothtb.v",
    "harris_response": "./harris_responsetb.v",
    "harris_threshold": "./harris_thresholdtb.v",
    "harris_corner": "./harris_cornertb.v",
}

def is_clocked_module(interface):
    return ("input clk" in interface) or ("input wire clk" in interface)

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./harris_corner_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)
    clk_signal = "clk" if is_clocked_module(interface) else None

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="Claude",
        clk_signal=clk_signal,
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,
    )

    results[filename] = result

    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

# Conv1D

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/conv_1d/shift_registertb.v ./shift_registertb.v
!cp ./ROME-LLM/testbenches/conv_1d/mac_unittb.v ./mac_unittb.v
!cp ./ROME-LLM/testbenches/conv_1d/conv1dtb.v ./conv1dtb.v

In [ ]:
#@title Submodules

### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["shift_register", "KERNEL_SIZE-stage shift register for sliding window with parameterizable data width and depth. On each valid_in, shifts data_in into tap[0] and older values shift right. Reset initializes all taps to 0 (zero-padding). Outputs a packed taps vector of width DATA_W*KERNEL_SIZE. Only shifts when valid_in is high.", "input clk, input rst, input valid_in, input [7:0] data_in, output [39:0] taps"],
    ["mac_unit", "Combinational 5-tap MAC unit with parameterizable kernel coefficients (default [2,8,12,8,2], sum=32). Takes newest sample (current data_in, not yet in shift register) and 4 previous taps from the shift register. Computes COEFF0*newest + COEFF1*tap[0] + COEFF2*tap[1] + COEFF3*tap[2] + COEFF4*tap[3], then right-shifts by GAIN_W=4 bits (floor division). This ensures zero-latency window formation: newest provides the current sample while taps provide the history.", "input [7:0] newest, input [39:0] taps, output [11:0] result"],
    ["conv1d", "1D streaming convolution filter with registered output. Instantiates shift_register and mac_unit. Uses causal same-length zero-padding: shift register is initialized to 0, so the first KERNEL_SIZE-1 outputs are boundary outputs computed with zero-padded partial windows. valid_out <= valid_in and data_out <= mac_result (both registered, 1-cycle delay). No pipeline latency — every valid_in produces a valid_out on the next cycle. Input N samples produces output N samples.", "input clk, input rst, input valid_in, input [7:0] data_in, output reg valid_out, output reg [11:0] data_out"],
]

In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical 1D convolution design
import os
import json

submodules = [
    ["shift_register", "KERNEL_SIZE-stage shift register for sliding window with parameterizable data width and depth. On each valid_in, shifts data_in into tap[0] and older values shift right. Reset initializes all taps to 0 (zero-padding). Outputs a packed taps vector of width DATA_W*KERNEL_SIZE. Only shifts when valid_in is high.", "input clk, input rst, input valid_in, input [7:0] data_in, output [39:0] taps"],
    ["mac_unit", "Combinational 5-tap MAC unit with parameterizable kernel coefficients (default [2,8,12,8,2], sum=32). Takes newest sample (current data_in, not yet in shift register) and 4 previous taps from the shift register. Computes COEFF0*newest + COEFF1*tap[0] + COEFF2*tap[1] + COEFF3*tap[2] + COEFF4*tap[3], then right-shifts by GAIN_W=4 bits (floor division). This ensures zero-latency window formation: newest provides the current sample while taps provide the history.", "input [7:0] newest, input [39:0] taps, output [11:0] result"],
    ["conv1d", "1D streaming convolution filter with registered output. Instantiates shift_register and mac_unit. Uses causal same-length zero-padding: shift register is initialized to 0, so the first KERNEL_SIZE-1 outputs are boundary outputs computed with zero-padded partial windows. valid_out <= valid_in and data_out <= mac_result (both registered, 1-cycle delay). No pipeline latency — every valid_in produces a valid_out on the next cycle. Input N samples produces output N samples.", "input clk, input rst, input valid_in, input [7:0] data_in, output reg valid_out, output reg [11:0] data_out"],
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

tb_paths = {
    "shift_register": "./shift_registertb.v",
    "mac_unit": "./mac_unittb.v",
    "conv1d": "./conv1dtb.v",
}

def is_clocked_module(interface):
    return ("input clk" in interface) or ("input wire clk" in interface)

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./conv1d_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)
    clk_signal = "clk" if is_clocked_module(interface) else None

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="Claude",
        clk_signal=clk_signal,
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,
    )

    results[filename] = result

    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

# Conv2D

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/conv_2d/line_buffertb.v ./line_buffertb.v
!cp ./ROME-LLM/testbenches/conv_2d/window_gen_2dtb.v ./window_gen_2dtb.v
!cp ./ROME-LLM/testbenches/conv_2d/mac_2dtb.v ./mac_2dtb.v
!cp ./ROME-LLM/testbenches/conv_2d/conv2dtb.v ./conv2dtb.v

In [ ]:
#@title Submodules

### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["line_buffer", "Shift-register FIFO line buffer with parameterizable depth IMG_WIDTH. Delays input by IMG_WIDTH cycles. Reset initializes all entries to 0. Only shifts when valid_in is high.", "input clk, input rst, input valid_in, input [7:0] data_in, output [7:0] data_out"],
    ["window_gen_2d", "2D window generator for streaming raster-order pixels. Uses 2 line_buffer instances (each IMG_WIDTH deep) and 3 row shift registers (3-tap each) to assemble a 3x3 pixel window. Supports PADDING_MODE parameter: 0=zero-padding (boundary pixels outside the image are treated as 0), 1=valid (no padding, window always contains actual pixel data only).", "input clk, input rst, input valid_in, input [7:0] pixel_in, output [71:0] window"],
    ["mac_2d", "Combinational 3x3 MAC unit. Multiplies 9 window pixels by fixed kernel coefficients [[1,2,1],[2,4,2],[1,2,1]] (sum=16), accumulates, and right-shifts by GAIN_W=4 bits (floor division by 16). All parameterizable.", "input [71:0] window, output [11:0] result"],
    ["conv2d", "2D streaming convolution with centered 3x3 kernel and registered output. Instantiates window_gen_2d and mac_2d. Supports PADDING_MODE parameter: 0=zero-padding (output size = input size), 1=valid (output size = (H-K+1)x(W-K+1), boundary rows/cols skipped). Pipeline latency is IMG_WIDTH+2 cycles plus 1 cycle for registered output.", "input clk, input rst, input valid_in, input [7:0] pixel_in, output reg valid_out, output reg [11:0] pixel_out"],
]


In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical 2D convolution design
import os
import json

submodules = [
    ["line_buffer", "Shift-register FIFO line buffer with parameterizable depth IMG_WIDTH. Delays input by IMG_WIDTH cycles. Reset initializes all entries to 0. Only shifts when valid_in is high.", "input clk, input rst, input valid_in, input [7:0] data_in, output [7:0] data_out"],
    ["window_gen_2d", "2D window generator for streaming raster-order pixels. Uses 2 line_buffer instances (each IMG_WIDTH deep) and 3 row shift registers (3-tap each) to assemble a 3x3 pixel window. Supports PADDING_MODE parameter: 0=zero-padding (boundary pixels outside the image are treated as 0), 1=valid (no padding, window always contains actual pixel data only).", "input clk, input rst, input valid_in, input [7:0] pixel_in, output [71:0] window"],
    ["mac_2d", "Combinational 3x3 MAC unit. Multiplies 9 window pixels by fixed kernel coefficients [[1,2,1],[2,4,2],[1,2,1]] (sum=16), accumulates, and right-shifts by GAIN_W=4 bits (floor division by 16). All parameterizable.", "input [71:0] window, output [11:0] result"],
    ["conv2d", "2D streaming convolution with centered 3x3 kernel and registered output. Instantiates window_gen_2d and mac_2d. Supports PADDING_MODE parameter: 0=zero-padding (output size = input size), 1=valid (output size = (H-K+1)x(W-K+1), boundary rows/cols skipped). Pipeline latency is IMG_WIDTH+2 cycles plus 1 cycle for registered output.", "input clk, input rst, input valid_in, input [7:0] pixel_in, output reg valid_out, output reg [11:0] pixel_out"],
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

tb_paths = {
    "line_buffer": "./line_buffertb.v",
    "window_gen_2d": "./window_gen_2dtb.v",
    "mac_2d": "./mac_2dtb.v",
    "conv2d": "./conv2dtb.v",
}

def is_clocked_module(interface):
    return ("input clk" in interface) or ("input wire clk" in interface)

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./conv2d_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)
    clk_signal = "clk" if is_clocked_module(interface) else None

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="ChatGPT",
        clk_signal=clk_signal,
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,
    )

    results[filename] = result

    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

# Conv3D

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/conv_3d/line_buffertb.v ./line_buffertb.v
!cp ./ROME-LLM/testbenches/conv_3d/window_gen_2dtb.v ./window_gen_2dtb.v
!cp ./ROME-LLM/testbenches/conv_3d/mac_3dtb.v ./mac_3dtb.v
!cp ./ROME-LLM/testbenches/conv_3d/conv3dtb.v ./conv3dtb.v

In [ ]:
#@title Submodules

### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["line_buffer", "Shift-register FIFO line buffer with parameterizable depth IMG_WIDTH. Delays input by IMG_WIDTH cycles. Reset initializes all entries to 0. Only shifts when valid_in is high. Reused as frame buffer by setting IMG_WIDTH=H*W.", "input clk, input rst, input valid_in, input [7:0] data_in, output [7:0] data_out"],
    ["window_gen_2d", "2D window generator for streaming raster-order pixels. Uses 2 line_buffer instances and 3 row shift registers to assemble a 3x3 pixel window. Supports PADDING_MODE parameter: 0=zero-padding (boundary pixels treated as 0), 1=valid (no padding). For conv3d, uses PADDING_MODE=1.", "input clk, input rst, input valid_in, input [7:0] pixel_in, output [71:0] window"],
    ["window_gen_3d", "3D window generator for streaming voxel data. Uses 2 frame buffers (line_buffer with depth=H*W) to delay by 1 and 2 frames, plus 3 window_gen_2d instances (PADDING_MODE=1) to form a 3x3 window per depth plane. Outputs packed 3x3x3 window (3 planes of 3x3 = 27 voxels). Plane order: plane0=2 frames ago, plane1=1 frame ago, plane2=current frame.", "input clk, input rst, input valid_in, input [7:0] voxel_in, output [215:0] window_3d"],
    ["mac_3d", "Combinational 3x3x3 MAC unit. Takes 27-voxel 3D window and 27-weight kernel (both externally packed), computes dot product (27 multiply-accumulate operations). Output width accommodates accumulation growth: DATA_W + ceil(log2(27)) = 13 bits for 8-bit input.", "input [215:0] window_3d, input [215:0] kernel, output [12:0] result"],
    ["conv3d", "Streaming 3D convolution with valid padding and registered output. Instantiates window_gen_3d and mac_3d. Pipeline latency = 2*H*W + W + 2 cycles (2 frame buffer fills + window_gen_2d fill). Kernel provided as external input port. Valid convolution: output skips boundary frames/rows/cols using row/col/frame counters. Output size: (D-K1+1)*(H-K2+1)*(W-K3+1).", "input clk, input rst, input [7:0] voxel_in, input valid_in, input [215:0] kernel, output reg [12:0] voxel_out, output reg valid_out"],
]


In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical 3D convolution design
import os
import json

submodules = [
    ["line_buffer", "Shift-register FIFO line buffer with parameterizable depth IMG_WIDTH. Delays input by IMG_WIDTH cycles. Reset initializes all entries to 0. Only shifts when valid_in is high. Reused as frame buffer by setting IMG_WIDTH=H*W.", "input clk, input rst, input valid_in, input [7:0] data_in, output [7:0] data_out"],
    ["window_gen_2d", "2D window generator for streaming raster-order pixels. Uses 2 line_buffer instances and 3 row shift registers to assemble a 3x3 pixel window. Supports PADDING_MODE parameter: 0=zero-padding (boundary pixels treated as 0), 1=valid (no padding). For conv3d, uses PADDING_MODE=1.", "input clk, input rst, input valid_in, input [7:0] pixel_in, output [71:0] window"],
    ["window_gen_3d", "3D window generator for streaming voxel data. Uses 2 frame buffers (line_buffer with depth=H*W) to delay by 1 and 2 frames, plus 3 window_gen_2d instances (PADDING_MODE=1) to form a 3x3 window per depth plane. Outputs packed 3x3x3 window (3 planes of 3x3 = 27 voxels). Plane order: plane0=2 frames ago, plane1=1 frame ago, plane2=current frame.", "input clk, input rst, input valid_in, input [7:0] voxel_in, output [215:0] window_3d"],
    ["mac_3d", "Combinational 3x3x3 MAC unit. Takes 27-voxel 3D window and 27-weight kernel (both externally packed), computes dot product (27 multiply-accumulate operations). Output width accommodates accumulation growth: DATA_W + ceil(log2(27)) = 13 bits for 8-bit input.", "input [215:0] window_3d, input [215:0] kernel, output [12:0] result"],
    ["conv3d", "Streaming 3D convolution with valid padding and registered output. Instantiates window_gen_3d and mac_3d. Pipeline latency = 2*H*W + W + 2 cycles (2 frame buffer fills + window_gen_2d fill). Kernel provided as external input port. Valid convolution: output skips boundary frames/rows/cols using row/col/frame counters. Output size: (D-K1+1)*(H-K2+1)*(W-K3+1).", "input clk, input rst, input [7:0] voxel_in, input valid_in, input [215:0] kernel, output reg [12:0] voxel_out, output reg valid_out"],
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

tb_paths = {
    "line_buffer": "./line_buffertb.v",
    "window_gen_2d": "./window_gen_2dtb.v",
    "window_gen_3d": "./window_gen_3dtb.v",
    "mac_3d": "./mac_3dtb.v",
    "conv3d": "./conv3dtb.v",
}

def is_clocked_module(interface):
    return ("input clk" in interface) or ("input wire clk" in interface)

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./conv3d_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)
    clk_signal = "clk" if is_clocked_module(interface) else None

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="ChatGPT",
        clk_signal=clk_signal,
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,
    )

    results[filename] = result

    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

# 16-bit MIPS Single-cycle Processor

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM

!cp ./ROME-LLM/testbenches/mips16-single-cycle/program_countertb.v ./program_countertb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/instruction_memorytb.v ./instruction_memorytb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/pc_addertb.v ./pc_addertb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/instruction_decodertb.v ./instruction_decodertb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/control_unittb.v ./control_unittb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/register_filetb.v ./register_filetb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/sign_extendtb.v ./sign_extendtb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/alu_control_unittb.v ./alu_control_unittb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/alutb.v ./alutb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/data_memorytb.v ./data_memorytb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/branch_address_addertb.v ./branch_address_addertb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/branch_controltb.v ./branch_controltb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/jump_address_logictb.v ./jump_address_logictb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/mux2_1_3bittb.v ./mux2_1_3bittb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/mux2_1_16bittb.v ./mux2_1_16bittb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/next_pc_logictb.v ./next_pc_logictb.v
!cp ./ROME-LLM/testbenches/mips16-single-cycle/mips16_single_cycletb.v ./mips16_single_cycletb.v

In [ ]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]

submodules = [
    ["program_counter", "Program_Counter",
     "input wire clk, input wire reset, input wire [15:0] pc_next, output reg [15:0] pc_current"],

    ["instruction_memory", "Instruction_Memory",
     "input wire [15:0] addr, output wire [15:0] instruction"],

    ["pc_adder", "PC_Adder",
     "input wire [15:0] pc_current, output wire [15:0] pc_plus_1"],

    ["instruction_decoder", "Instruction_Decoder",
     "input wire [15:0] instruction, output wire [3:0] opcode, output wire [2:0] rs, output wire [2:0] rt, output wire [2:0] rd, output wire [2:0] funct, output wire [6:0] imm7, output wire [11:0] jump_addr"],

    ["control_unit", "Control_Unit",
     "input wire [3:0] opcode, output wire reg_dst, output wire alu_src, output wire mem_to_reg, output wire reg_write, output wire mem_read, output wire mem_write, output wire branch, output wire jump, output wire [1:0] alu_op"],

    ["register_file", "Register_File",
     "input wire clk, input wire reset, input wire reg_write, input wire [2:0] read_reg1, input wire [2:0] read_reg2, input wire [2:0] write_reg, input wire [15:0] write_data, output wire [15:0] read_data1, output wire [15:0] read_data2"],

    ["sign_extend", "Sign_Extend",
     "input wire [6:0] imm_in, output wire [15:0] imm_out"],

    ["alu_control_unit", "ALU_Control_Unit",
     "input wire [1:0] alu_op, input wire [2:0] funct, output wire [2:0] alu_ctrl"],

    ["alu", "ALU",
     "input wire [15:0] a, input wire [15:0] b, input wire [2:0] alu_ctrl, output reg [15:0] result, output wire zero"],

    ["data_memory", "Data_Memory",
     "input wire clk, input wire mem_read, input wire mem_write, input wire [15:0] addr, input wire [15:0] write_data, output wire [15:0] read_data"],

    ["branch_address_adder", "Branch_Address_Adder",
     "input wire [15:0] pc_plus_1, input wire [15:0] imm_ext, output wire [15:0] branch_addr"],

    ["branch_control", "Branch_Control",
     "input wire branch, input wire zero, output wire pc_src"],

    ["jump_address_logic", "Jump_Address_Logic",
     "input wire [15:0] pc_plus_1, input wire [11:0] jump_addr, output wire [15:0] jump_target"],

    ["mux2_1_3bit", "mux2_1_3bit",
     "input wire [2:0] in0, input wire [2:0] in1, input wire sel, output wire [2:0] out"],

    ["mux2_1_16bit", "mux2_1_16bit",
     "input wire [15:0] in0, input wire [15:0] in1, input wire sel, output wire [15:0] out"],

    ["next_pc_logic", "Next_PC_Logic",
     "input wire [15:0] pc_plus_1, input wire [15:0] branch_addr, input wire [15:0] jump_target, input wire pc_src, input wire jump, output wire [15:0] pc_next"]
]

In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical 16-bit MIPS-like CPU design
import os
import json

submodules = [
    ["program_counter", "Program_Counter",
     "input wire clk, input wire reset, input wire [15:0] pc_next, output reg [15:0] pc_current"],

    ["instruction_memory", "Instruction_Memory",
     "input wire [15:0] addr, output wire [15:0] instruction"],

    ["pc_adder", "PC_Adder",
     "input wire [15:0] pc_current, output wire [15:0] pc_plus_1"],

    ["instruction_decoder", "Instruction_Decoder",
     "input wire [15:0] instruction, output wire [3:0] opcode, output wire [2:0] rs, output wire [2:0] rt, output wire [2:0] rd, output wire [2:0] funct, output wire [6:0] imm7, output wire [11:0] jump_addr"],

    ["control_unit", "Control_Unit",
     "input wire [3:0] opcode, output wire reg_dst, output wire alu_src, output wire mem_to_reg, output wire reg_write, output wire mem_read, output wire mem_write, output wire branch, output wire jump, output wire [1:0] alu_op"],

    ["register_file", "Register_File",
     "input wire clk, input wire reset, input wire reg_write, input wire [2:0] read_reg1, input wire [2:0] read_reg2, input wire [2:0] write_reg, input wire [15:0] write_data, output wire [15:0] read_data1, output wire [15:0] read_data2"],

    ["sign_extend", "Sign_Extend",
     "input wire [6:0] imm_in, output wire [15:0] imm_out"],

    ["alu_control_unit", "ALU_Control_Unit",
     "input wire [1:0] alu_op, input wire [2:0] funct, output wire [2:0] alu_ctrl"],

    ["alu", "ALU",
     "input wire [15:0] a, input wire [15:0] b, input wire [2:0] alu_ctrl, output reg [15:0] result, output wire zero"],

    ["data_memory", "Data_Memory",
     "input wire clk, input wire mem_read, input wire mem_write, input wire [15:0] addr, input wire [15:0] write_data, output wire [15:0] read_data"],

    ["branch_address_adder", "Branch_Address_Adder",
     "input wire [15:0] pc_plus_1, input wire [15:0] imm_ext, output wire [15:0] branch_addr"],

    ["branch_control", "Branch_Control",
     "input wire branch, input wire zero, output wire pc_src"],

    ["jump_address_logic", "Jump_Address_Logic",
     "input wire [15:0] pc_plus_1, input wire [11:0] jump_addr, output wire [15:0] jump_target"],

    ["mux2_1_3bit", "mux2_1_3bit",
     "input wire [2:0] in0, input wire [2:0] in1, input wire sel, output wire [2:0] out"],

    ["mux2_1_16bit", "mux2_1_16bit",
     "input wire [15:0] in0, input wire [15:0] in1, input wire sel, output wire [15:0] out"],

    ["next_pc_logic", "Next_PC_Logic",
     "input wire [15:0] pc_plus_1, input wire [15:0] branch_addr, input wire [15:0] jump_target, input wire pc_src, input wire jump, output wire [15:0] pc_next"]
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

tb_paths = {
    "program_counter": "./program_countertb.v",
    "instruction_memory": "./instruction_memorytb.v",
    "pc_adder": "./pc_addertb.v",
    "instruction_decoder": "./instruction_decodertb.v",
    "control_unit": "./control_unittb.v",
    "register_file": "./register_filetb.v",
    "sign_extend": "./sign_extendtb.v",
    "alu_control_unit": "./alu_control_unittb.v",
    "alu": "./alutb.v",
    "data_memory": "./data_memorytb.v",
    "branch_address_adder": "./branch_address_addertb.v",
    "branch_control": "./branch_controltb.v",
    "jump_address_logic": "./jump_address_logictb.v",
    "mux2_1_3bit": "./mux2_1_3bittb.v",
    "mux2_1_16bit": "./mux2_1_16bittb.v",
    "next_pc_logic": "./next_pc_logictb.v",
}

def is_clocked_module(interface):
    return ("input wire clk" in interface) or ("input clk" in interface)

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./mips16_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)
    clk_signal = "clk" if is_clocked_module(interface) else None

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="ChatGPT",
        clk_signal=clk_signal,
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,
    )

    results[filename] = result

    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

# Zip files and such

In [ ]:
print("yes")

In [ ]:
#@title Zip the whole thing
!zip -r /content/zipitall.zip /content

In [ ]:
!zip -r /content/conv1dfiles1.zip /content/conv1d /content/conv1d_ppa_runs /content/mac_unit /content/shift_register